<a href="https://colab.research.google.com/github/vanessalopc/learning-linux-command-line/blob/main/AI_for_GxE_playground.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI for GxE demo notebook

**Created by Aike Potze, PhD candidate at Wageningen University for the AAGI 2026 workshop, *AI for GxE***.

Welcome to the demo notebook for the AI for GxE workshop. In this notebook, you will train your own very simple neural networks for the prediction of yield based on genotype, environment and management variables.

As dataset we will utilize the [BRIWECS](https://doi.org/10.1038/s41597-024-04332-7) dataset (Wang et al., 2025). It is a German wheat yield dataset with very comprehensive replication between management practices, environments, and genotypes. From section 4 onwards, we heavily mask the genotype-environment combinations to create a challenging, imbalanced multi-environment trial with a shifting population of genotypes over time, which is more reflective of real-world breeding scenarios than complete replication.

Wang, T. C., Rose, T., Zetzsche, H., Ballvora, A., Friedt, W., Kage, H., ... & Chen, T. W. (2025). Multi-environment field trials for wheat yield, stability and breeding progress in Germany. Scientific data, 12(1), 64.

The ultimate goal of this demo is to predict the wheat yield and future rankings of specific genotype-environment combinations based on the available genotype, environment, and management data. The genotype-environment combinations are new genotypes in the final year of a multi-environment trial. Finally, the goal is to predict the genotype differences within each test environment correctly, as this corresponds to selecting the best genotypes for each environment.

This notebook is specifically designed to highlight the challenge of ranking prediction under this exact scenario.

**Overall instructions:** Select the correct runtime and run all cells. Wait for the dataset-preparation cell to finish before expecting its menus and the training controls; the first setup downloads and parses a roughly 105 MB SNP workbook and may take several minutes. No coding should be required, and the backend code (collapsed cells) is not intended for direct user interaction.

**Colab instructions:** A working GPU is required. Select **Runtime > Change runtime type > Hardware accelerator > GPU** (for example, T4), save, reconnect, and **Run all**. Setup checks that PyTorch can run on the GPU before downloading the dataset. If Colab cannot allocate a GPU, try again later or run locally, where CPU training is supported. Internet is required for data downloads.


In [ ]:
#@title Debugging quick setup (leave off for the workshop)
debug_dummy_snps = False #@param {type:"boolean"}

if debug_dummy_snps:
    print("Debug mode: dummy SNPs replace the workbook; real field/weather data still load. "
          "Cohort membership may differ. Results are for debugging only.")


## 00 - Dataset description

BRIWECS covers 228 winter-wheat cultivars at six German locations. This notebook has two independent model-training tracks. Both predict **new genotypes in a new year**: training uses 2015-2017, validation uses held-out genotypes in 2018, and testing uses different held-out genotypes in 2019. Other records of validation and test genotypes are excluded from model fitting and scoring.

| Dataset | Playground section | Diagnosing issues section | Test comparison section | Records and genotype split |
|---|---|---|---|---|
| **Complete dataset** | **Section 01** | **Section 02** | **Section 03** | All available training-genotype records in 2015-2017, without simulated cohort masking. Randomly assign approximately 70% of genotypes to training, 15% to validation, and 15% to testing (seed 42). Holdout sampling requires observations in its evaluation year. |
| **Sparse cohort dataset** | **Section 04** | **Section 05** | **Section 06** | Existing simulated 2015-2019 introduction cohorts, with sparse one-/two-year coverage. Earlier cohorts train, the 2018 cohort validates, and the 2019 cohort tests. This split is unchanged. |

The two tracks have the same controls and metrics but separate saved models. Their test genotypes and coverage differ, so compare models within each track rather than interpreting cross-dataset scores as a paired benchmark. The **Genotype × trial** menu shows both designs and the original observations. Other exploratory views omit test genotypes from both splits to keep their outcomes hidden. SNP-based models exclude genotypes without SNP data.


In [ ]:
#@title Prepare interactive visuals
import importlib.util
import os
import sys
from importlib.metadata import PackageNotFoundError, version

in_colab = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
try:
    plotly_version = version("plotly")
    plotly_major = int(plotly_version.split(".")[0])
except PackageNotFoundError:
    plotly_version = "not installed"
    plotly_major = 0
required_modules = (("ipywidgets", "openpyxl", "scipy", "nest_asyncio") if in_colab
                    else ("anywidget", "ipywidgets", "openpyxl", "scipy"))
missing_modules = [name for name in required_modules
                   if importlib.util.find_spec(name) is None]
try:
    widget_major = int(version("ipywidgets").split(".")[0])
except PackageNotFoundError:
    widget_major = 0
missing_visuals = plotly_major != 6 or bool(missing_modules) or (in_colab and widget_major != 7)
if in_colab and missing_visuals:
    print("Preparing interactive visuals for Colab...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "plotly>=6,<7", "ipywidgets>=7,<8", "openpyxl", "scipy", "nest_asyncio"])
    importlib.invalidate_caches()
    loaded_widgets = sys.modules.get("ipywidgets")
    if loaded_widgets is not None and int(loaded_widgets.__version__.split(".")[0]) != 7:
        raise RuntimeError("ipywidgets 7 is installed, but an incompatible version is still loaded. "
                           "Restart the Colab session and Run all before using the controls.")
elif missing_visuals:
    missing = ([f"plotly 6.x (found {plotly_version})"] if plotly_major != 6 else [])
    missing.extend(missing_modules)
    raise ImportError(
        f"Missing notebook dependencies in the active Python kernel ({sys.executable}): "
        f"{', '.join(missing)}. Install the missing notebook dependencies in this Python environment "
        "and restart the kernel.")
if in_colab:
    from google.colab import output
    output.disable_custom_widget_manager()
    print("Configured Colab's native widget manager.", flush=True)

#@title Load and prepare the BRIWECS data
import csv
import html
import hashlib
import io
import os
import sys
import urllib.error
import urllib.request
import zipfile
from collections import defaultdict
from datetime import date, timedelta
import calendar

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import torch
from torch import nn
import ipywidgets as widgets
from IPython.display import HTML, display
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from contextlib import contextmanager

class ColabPlot(widgets.HTML):
    """Update embedded Plotly HTML without output capture or layout synchronization."""
    def __init__(self, figure=None):
        super().__init__(layout=widgets.Layout(width="100%"))
        self._figure = go.Figure(figure)

    def __getattr__(self, name):
        figure = self.__dict__.get("_figure")
        if figure is not None:
            return getattr(figure, name)
        raise AttributeError(name)

    def refresh(self):
        height = int(self._figure.layout.height or 450)
        self.layout.max_width = f"{2 * height}px"
        document = self._figure.to_html(include_plotlyjs="cdn", full_html=True,
                                        config={"responsive": True})
        self.value = (f'<iframe title="Interactive Plotly chart" srcdoc="{html.escape(document, quote=True)}" '
                      f'style="width:100%;max-width:{2 * height}px;height:{height + 25}px;border:0"></iframe>')

    @contextmanager
    def batch_update(self):
        with self._figure.batch_update():
            yield self
        self.refresh()

plot_width_style = widgets.HTML()
plot_width_rules = {}

def limit_native_plot_width(figure):
    previous_class = None
    def update_width(layout, height):
        nonlocal previous_class
        height = int(height or 450)
        class_name = f"briwecs-plot-height-{height}"
        if previous_class is not None:
            figure.remove_class(previous_class)
        figure.add_class(class_name)
        previous_class = class_name
        plot_width_rules[class_name] = f".{class_name} {{ width: 100%; max-width: {2 * height}px; }}"
        plot_width_style.value = "<style>" + "".join(plot_width_rules.values()) + "</style>"
    figure.layout.on_change(update_width, "height")
    update_width(figure.layout, figure.layout.height)
    return figure

def figure_widget(figure=None):
    if in_colab:
        widget = ColabPlot(figure)
        if figure is not None:
            widget.refresh()
        return widget
    return limit_native_plot_width(go.FigureWidget(figure))

def refresh_colab_plots(widget):
    if isinstance(widget, ColabPlot):
        widget.refresh()
    else:
        for child in getattr(widget, "children", ()):
            refresh_colab_plots(child)

def plot_layout(figure):
    return figure._figure.layout if isinstance(figure, ColabPlot) else figure.layout

def display_panel(widget):
    if not in_colab:
        display(plot_width_style)
    display(widget)
    refresh_colab_plots(widget)

torch.set_num_threads(1)

def select_training_device():
    gpu_instructions = (
        "Select Runtime > Change runtime type > Hardware accelerator > GPU (for example, T4), "
        "save, reconnect, and Run all. If Colab cannot allocate a GPU, try again later "
        "or run locally, where CPU training is supported.")
    if torch.cuda.is_available():
        selected = torch.device("cuda")
        try:
            probe = torch.ones((2, 2), device=selected)
            probe @ probe
            torch.cuda.synchronize(selected)
        except RuntimeError as exc:
            raise RuntimeError("PyTorch could not run on the CUDA GPU. " + gpu_instructions) from exc
        print(f"Neural-network device: {selected} ({torch.cuda.get_device_name(selected)}); "
              "GPU computation verified.", flush=True)
        return selected
    if in_colab:
        raise RuntimeError("This Colab workshop requires a CUDA GPU, but PyTorch cannot access one. "
                           + gpu_instructions)
    selected = (torch.device("mps")
                if hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
                else torch.device("cpu"))
    print(f"Neural-network device: {selected}" +
          (" (local CPU training is supported but slower)" if selected.type == "cpu" else ""),
          flush=True)
    return selected

device = select_training_device()

def device_tensor(*args, **kwargs):
    return torch.tensor(*args, **kwargs, device=device)
plt.rcParams.update({"figure.facecolor": "white", "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})

# Optional offline use: set BRIWECS_ZIP to the original Figshare archive.
archive_path = os.environ.get("BRIWECS_ZIP")
if archive_path:
    with open(archive_path, "rb") as source:
        archive_bytes = source.read()
else:
    print("Downloading the original BRIWECS archive (11.3 MB)...")
    request = urllib.request.Request(
        "https://ndownloader.figshare.com/files/50832927",
        headers={"User-Agent": "BRIWECS-workshop-notebook/1.0"})
    with urllib.request.urlopen(request, timeout=120) as response:
        archive_bytes = response.read()

expected_md5 = "fb6543627c67a368b4fe5f078757f8b5"
if hashlib.md5(archive_bytes).hexdigest() != expected_md5:
    raise ValueError("BRIWECS archive checksum does not match the published dataset.")

weather_variables = {
    "Rainfall (mm)": ("Precipitation_mm_d", "sum"),
    "Mean air temperature (°C)": ("Temperature.Air.Mean_C", "mean"),
    "Mean relative humidity (%)": ("Relative.Air.Humidity_percent", "mean"),
    "Mean daily solar radiation (MJ/m²/day)": ("Global.Radiation_MJ_m2_d", "mean"),
    "Mean wind speed (m/s)": ("Wind.Speed_m_s", "mean"),
    "Minimum air temperature (°C)": ("Temperature.Air.Min_C", "min"),
    "Maximum air temperature (°C)": ("Temperature.Air.Max_C", "max"),
}
weather_columns = {f"Month {month} after sowing: {label}": (column, reduction)
                   for month in range(1, 10)
                   for label, (column, reduction) in weather_variables.items()}
trait_label = "Grain yield (dt/ha, dry mass)"
trait_columns = {
    trait_label: "Seedyield",
    "Grain protein (%)": "Crude_protein",
    "Thousand grain weight (g)": "TGW",
    "Plant height (cm)": "Plantheight_bio",
    "Biomass (dt/ha)": "Biomass",
    "Sedimentation (ml)": "Sedimentation",
}
weather_days = defaultdict(dict)
yield_groups = defaultdict(list)
trait_groups = defaultdict(lambda: defaultdict(list))
sowing_days = defaultdict(list)

def month_anniversary(start, offset):
    month_index = start.year * 12 + start.month - 1 + offset
    year, month_zero = divmod(month_index, 12)
    month = month_zero + 1
    return date(year, month, min(start.day, calendar.monthrange(year, month)[1]))

with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    weather_files = sorted(name for name in archive.namelist()
                           if "/data/weather/DWD_Intpol_2014_2024_" in name
                           and name.endswith(".csv"))
    if len(weather_files) != 6:
        raise ValueError(f"Expected six weather files, found {len(weather_files)}.")
    for name in weather_files:
        with archive.open(name) as raw:
            for row in csv.DictReader(io.TextIOWrapper(raw, encoding="cp1252"), delimiter=";"):
                day = date.fromisoformat(row["Date"])
                location = row["Location"]
                if day in weather_days[location]:
                    raise ValueError(f"Duplicate weather date for {location}: {day}")
                values = [float(row[column].replace(",", "."))
                          for column, _ in weather_variables.values()]
                if not np.all(np.isfinite(values)):
                    raise ValueError(f"Non-finite weather for {location}: {day}")
                weather_days[location][day] = values

    with archive.open("Briwecs_data/output/BRIWECS_data_publication.csv") as raw:
        for row in csv.DictReader(io.TextIOWrapper(raw, encoding="cp1252"), delimiter=";"):
            location, year, management = row["Location"], int(row["Year"]), row["Treatment"]
            trial = (location, year, management)
            sowing = row["Sowing_date"].strip()
            if sowing and sowing != "NA":
                try:
                    day_of_year = int(sowing)
                except ValueError as exc:
                    raise ValueError(f"Invalid sowing day for {trial}: {sowing}") from exc
                max_day = 366 if calendar.isleap(year - 1) else 365
                if not 1 <= day_of_year <= max_day:
                    raise ValueError(f"Invalid sowing day for {trial}: {sowing}")
                sowing_days[trial].append(day_of_year)
            key = (location, year, management, row["BRISONr"])
            for label, column in trait_columns.items():
                value = row[column].strip()
                if value and value != "NA":
                    number = float(value.replace(",", "."))
                    if not np.isfinite(number):
                        raise ValueError(f"Non-finite {column} value for {key}")
                    trait_groups[key][label].append(number)
                    if label == trait_label:
                        yield_groups[key].append(number)

trial_weather = {}
for location, year, management in sorted({key[:3] for key in yield_groups}):
    trial = (location, year, management)
    if not sowing_days[trial]:
        raise ValueError(f"No sowing date available for trial {trial}.")
    median_day = int(np.floor(np.median(sowing_days[trial])))
    sowing_date = date(year - 1, 1, 1) + timedelta(days=median_day - 1)
    features = {}
    for month in range(1, 10):
        start = month_anniversary(sowing_date, month - 1)
        end = month_anniversary(sowing_date, month)
        days = [start + timedelta(days=i) for i in range((end - start).days)]
        missing = [day for day in days if day not in weather_days[location]]
        if missing:
            raise ValueError(f"Incomplete weather for {trial}, month {month}: "
                             f"{len(missing)} missing days, first {missing[0]}.")
        daily = np.asarray([weather_days[location][day] for day in days], dtype=float)
        for i, (label, (_, reduction)) in enumerate(weather_variables.items()):
            reducers = {"sum": np.sum, "mean": np.mean, "min": np.min, "max": np.max}
            features[f"Month {month} after sowing: {label}"] = float(reducers[reduction](daily[:, i]))
    trial_weather[trial] = (sowing_date, features)

observations = []
for (location, year, management, genotype), yields in sorted(yield_groups.items()):
    sowing_date, ec = trial_weather[(location, year, management)]
    observations.append({"location": location, "year": year, "management": management, "genotype": genotype,
                         "sowing_date": sowing_date, "yield": float(np.mean(yields)),
                         "plots": len(yields),
                         "traits": {label: float(np.mean(values))
                                    for label, values in trait_groups[(location, year, management, genotype)].items()},
                         **ec})

if not observations:
    raise ValueError("No yield observations found in the BRIWECS archive.")
print(f"Ready: {sum(item['plots'] for item in observations):,} yield records "
      f"combined into {len(observations)} cultivar × trial means.")


# Define the simulated sparse cohorts before selecting any features.
import http.client
import time
import xml.etree.ElementTree as ET

def snp_workbook_from_download(payload):
    if not zipfile.is_zipfile(io.BytesIO(payload)):
        detail = payload[:500].decode("utf-8", errors="replace")
        raise ValueError(f"SNP source returned an error page or incomplete ZIP: {detail}")
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        if "DataSheet_2.xlsx" in archive.namelist():
            workbook_bytes = archive.read("DataSheet_2.xlsx")
        elif "xl/worksheets/sheet1.xml" in archive.namelist():
            workbook_bytes = payload
        else:
            raise ValueError("SNP archive does not contain DataSheet_2.xlsx or its Excel workbook.")
    if not zipfile.is_zipfile(io.BytesIO(workbook_bytes)):
        raise ValueError("Downloaded SNP workbook is not a valid XLSX file.")
    with zipfile.ZipFile(io.BytesIO(workbook_bytes)) as workbook:
        required = {"xl/worksheets/sheet1.xml", "xl/sharedStrings.xml"}
        if not required.issubset(workbook.namelist()):
            raise ValueError("Downloaded SNP workbook is missing the expected Excel data.")
    # Checksum published in the PMC7019858.1 cloud metadata.
    if hashlib.md5(workbook_bytes).hexdigest() != "53eddaca86cf51e3ac07a04772868c88":
        raise ValueError("SNP workbook checksum does not match the original supplement.")
    return workbook_bytes

if debug_dummy_snps:
    snp_ids = sorted({row["genotype"] for row in observations})
    print("DEBUG: synthetic SNPs; skipping workbook download and parsing. "
          "SNP-based results are not biological evidence.", flush=True)
else:
    path = os.environ.get("BRIWECS_SNP_ZIP")
    if path:
        with open(path, "rb") as source:
            snp_workbook_bytes = snp_workbook_from_download(source.read())
    else:
        print("Downloading SNP workbook from PMC (~105 MB)...", flush=True)
        for attempt in range(1, 4):
            try:
                url = ("https://pmc-oa-opendata.s3.amazonaws.com/PMC7019858.1/DataSheet_2.xlsx"
                       if attempt < 3 else
                       "https://www.ebi.ac.uk/europepmc/webservices/rest/PMC7019858/supplementaryFiles")
                request = urllib.request.Request(
                    url,
                    headers={"User-Agent": "BRIWECS-workshop-notebook/1.0"})
                with urllib.request.urlopen(request, timeout=300) as response:
                    snp_workbook_bytes = snp_workbook_from_download(response.read())
                break
            except (urllib.error.URLError, http.client.HTTPException, TimeoutError,
                    ConnectionError, ValueError, zipfile.BadZipFile) as exc:
                print(f"SNP download attempt {attempt}/3 failed: {exc}", flush=True)
                if attempt == 3:
                    raise RuntimeError(
                        "The SNP download sources are unavailable or returned incomplete data. "
                        "Download Supplementary Data Sheet 2 from "
                        "https://doi.org/10.3389/fpls.2019.01771 and set BRIWECS_SNP_ZIP "
                        "to the local DataSheet_2.xlsx or supplement ZIP before running again.") from exc
                time.sleep(2 * attempt)
    namespace = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
    with zipfile.ZipFile(io.BytesIO(snp_workbook_bytes)) as workbook:
        with workbook.open("xl/worksheets/sheet1.xml") as source:
            for _, row in ET.iterparse(source, events=("end",)):
                if row.tag == namespace + "row":
                    snp_ids = [f"BRISONr_{int(c.find(namespace + 'v').text)}" for c in list(row)[11:]]
                    break
    if len(snp_ids) != 221 or len(set(snp_ids)) != len(snp_ids):
        raise ValueError("Unexpected SNP sample identifiers.")
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    with archive.open("Briwecs_data/metadata/BRIWECs_cultivar_info.csv") as raw:
        cultivar_info = list(csv.DictReader(io.TextIOWrapper(raw, encoding="cp1252")))
observed_ids = {row["genotype"] for row in observations if 2015 <= row["year"] <= 2019}
release_info = {row["BRISONr"]: (int(row["RYear"]), row["genotype"])
                for row in cultivar_info if row["RYear"].isdigit()
                and row["BRISONr"] in snp_ids and row["BRISONr"] in observed_ids}
ordered_cultivars = sorted(release_info, key=lambda g: (release_info[g][0], g))
if len(ordered_cultivars) < 10:
    raise ValueError("Not enough cultivars with SNPs and release years for five cohorts.")
years_observed = defaultdict(set)
for row in observations:
    years_observed[row["genotype"]].add(row["year"])
cohort_size = len(ordered_cultivars) // 5
test_order = [g for g in reversed(ordered_cultivars) if 2019 in years_observed[g]][:cohort_size]
remaining = [g for g in ordered_cultivars if g not in set(test_order)]
validation_order = [g for g in reversed(remaining) if 2018 in years_observed[g]][:cohort_size]
if len(test_order) != cohort_size or len(validation_order) != cohort_size:
    raise ValueError("Not enough cultivars with observed yields in the held-out years.")
remaining = [g for g in remaining if g not in set(validation_order)]
groups = list(np.array_split(remaining, 3)) + [validation_order, test_order]
cohort_year = {g: 2015 + i for i, group in enumerate(groups) for g in group}
rng = np.random.default_rng(42)
two_year_genotypes = set()
for i, group in enumerate(groups):
    year = 2015 + i
    candidates = [g for g in group if year in years_observed[g] and year + 1 in years_observed[g]]
    count = round(len(group) / 3)
    if len(candidates) < count:
        raise ValueError(f"Not enough cultivars observed in both years for cohort {year}.")
    two_year_genotypes.update(rng.choice(sorted(candidates), size=count, replace=False).tolist())
train_genotypes = {g for g, year in cohort_year.items() if year <= 2017}
val_genotypes = {g for g, year in cohort_year.items() if year == 2018}
test_genotypes = {g for g, year in cohort_year.items() if year == 2019}
genotypes = ordered_cultivars

def cohort_eligible(row):
    start = cohort_year[row["genotype"]]
    return row["year"] == start or (row["genotype"] in two_year_genotypes and row["year"] == start + 1)

def cohort_split(row):
    if row["genotype"] not in cohort_year:
        return "Not used"
    cohort, year = cohort_year[row["genotype"]], row["year"]
    if cohort <= 2017 and 2015 <= year <= 2017:
        return "Train"
    if cohort == 2018 and year == 2018:
        return "Validation"
    if cohort == 2019 and year == 2019:
        return "Test"
    return "Not used"

def split_name(row):
    if row["genotype"] not in cohort_year or not cohort_eligible(row):
        return "Not used"
    return cohort_split(row)

def cohort_rows(sparse):
    return [r for r in observations if r["genotype"] in cohort_year
            and cohort_split(r) != "Not used" and (not sparse or cohort_eligible(r))]

def cohort_decomp_rows(sparse):
    return [r for r in observations if r["genotype"] in cohort_year
            and cohort_year[r["genotype"]] <= 2018 and 2015 <= r["year"] <= 2018
            and (not sparse or cohort_eligible(r))]

assert train_genotypes.isdisjoint(val_genotypes | test_genotypes)
assert val_genotypes.isdisjoint(test_genotypes)
print(f"Sparse cohort design: {len(ordered_cultivars)} cultivars; "
      f"{sum(split_name(r) != 'Not used' for r in observations):,} model observations.")


def random_genotype_groups(rows, seed=42):
    available = defaultdict(set)
    for row in rows:
        if 2015 <= row["year"] <= 2019:
            available[row["genotype"]].add(row["year"])
    identifiers = sorted(available)
    holdout_size = round(len(identifiers) * 0.15)
    if holdout_size < 1:
        raise ValueError("At least four genotypes are needed for random 15% holdouts.")
    random = np.random.default_rng(seed)
    # Reserve validation-only candidates before sampling test genotypes.
    validation_candidates = [g for g in identifiers if 2018 in available[g]]
    test_candidates = [g for g in identifiers if 2019 in available[g]]
    if len(validation_candidates) < holdout_size or len(test_candidates) < holdout_size:
        raise ValueError("Not enough genotypes observed in the validation and test years.")
    validation_only = [g for g in validation_candidates if g not in set(test_candidates)]
    overlap_needed = max(0, holdout_size - len(validation_only))
    overlap = sorted(set(validation_candidates) & set(test_candidates))
    if len(test_candidates) - overlap_needed < holdout_size:
        raise ValueError("Not enough distinct genotypes for both year-specific holdouts.")
    reserved = set(random.choice(overlap, size=overlap_needed, replace=False).tolist())
    test = set(random.choice(
        [g for g in test_candidates if g not in reserved], size=holdout_size, replace=False).tolist())
    validation = reserved | set(random.choice(
        [g for g in validation_candidates if g not in test and g not in reserved],
        size=holdout_size - len(reserved), replace=False).tolist())
    return set(identifiers) - test - validation, validation, test

complete_train_genotypes, complete_val_genotypes, complete_test_genotypes = random_genotype_groups(observations)

def complete_split_name(row):
    genotype, year = row["genotype"], row["year"]
    if genotype in complete_train_genotypes and 2015 <= year <= 2017:
        return "Train"
    if genotype in complete_val_genotypes and year == 2018:
        return "Validation"
    if genotype in complete_test_genotypes and year == 2019:
        return "Test"
    return "Not used"

heldout_test_genotypes = test_genotypes | complete_test_genotypes
print(f"Complete design: {len(complete_train_genotypes)} training, "
      f"{len(complete_val_genotypes)} validation, {len(complete_test_genotypes)} test genotypes; "
      f"{sum(complete_split_name(r) != 'Not used' for r in observations):,} model observations.")

#@title Explore cultivar × trial coverage
import html

split_colors = {"Train": "#11877f", "Validation": "#e9a23b", "Test": "#e05c71"}
site_names = {
    "GGE": "Gross Gerau", "HAN": "Hannover", "KAL": "Klein Altendorf",
    "KIE": "Kiel", "QLB": "Quedlinburg", "RHH": "Rauischholzhausen",
}
exploration_observations = [r for r in observations if r["genotype"] not in heldout_test_genotypes]
trials = sorted({(r["year"], r["location"], r["management"]) for r in exploration_observations})
cultivar_names = {r["BRISONr"]: r["genotype"] for r in cultivar_info}
subset_observations = [r for r in observations
                       if r["genotype"] in cohort_year and cohort_eligible(r)]
last_sparse_year = {}
for row in subset_observations:
    genotype = row["genotype"]
    last_sparse_year[genotype] = max(last_sparse_year.get(genotype, row["year"]), row["year"])
ordered_genotypes = sorted(train_genotypes | val_genotypes | test_genotypes,
    key=lambda g: (last_sparse_year[g], cohort_year[g], int(g.split("_")[1])))
exploration_subset = [r for r in subset_observations if r["genotype"] not in heldout_test_genotypes]
exploration_genotypes = [g for g in ordered_genotypes if g not in heldout_test_genotypes]

def landscape_figure(subsetted, complete=False):
    rows = ([r for r in observations if 2015 <= r["year"] <= 2019]
            if complete else subset_observations if subsetted else observations)
    trials = sorted({(r["year"], r["location"], r["management"])
                     for r in (rows if complete else observations)})
    splitter = complete_split_name if complete else split_name
    lookup = {(r["genotype"], (r["year"], r["location"], r["management"])): r for r in rows}
    genotype_order = ordered_genotypes if subsetted else ordered_genotypes + sorted(
        {r["genotype"] for r in observations} - set(ordered_genotypes),
        key=lambda g: int(g.split("_")[1]))
    if complete:
        genotype_order = (sorted(complete_train_genotypes) + sorted(complete_val_genotypes)
                          + sorted(complete_test_genotypes))
    codes = {"Missing": 0, "Not used for model": 1,
             "Train": 2, "Validation": 3, "Test": 4, "Observed": 2}
    matrix, hover = [], []
    for genotype in genotype_order:
        row_values, row_hover = [], []
        for trial in trials:
            row = lookup.get((genotype, trial))
            category = "Missing"
            if row:
                category = splitter(row) if subsetted else "Observed"
                if category == "Not used":
                    category = "Not used for model"
            row_values.append(codes[category])
            name = cultivar_names.get(genotype, genotype)
            row_hover.append(
                f"{html.escape(name)} · {genotype}<br>{trial[0]} · {site_names[trial[1]]}"
                f"<br>{trial[2]}<br><b>{category}</b>"
                + ("<br>Yield withheld for final evaluation" if row and genotype in heldout_test_genotypes
                   else f"<br>Yield: {row['yield']:.1f} dt/ha · {row['plots']} plots" if row else ""))
        matrix.append(row_values)
        hover.append(row_hover)
    figure = go.Figure(go.Heatmap(
        z=matrix, customdata=hover, x=list(range(len(trials))), y=genotype_order,
        zmin=0, zmax=4, xgap=0, ygap=0, showscale=False,
        colorscale=[[0, "#e4e9ec"], [.1249, "#e4e9ec"], [.125, "#c7d9e7"], [.3749, "#c7d9e7"],
                    [.375, "#11877f"], [.6249, "#11877f"], [.625, "#e9a23b"], [.8749, "#e9a23b"],
                    [.875, "#e05c71"], [1, "#e05c71"]],
        hovertemplate="%{customdata}<extra></extra>"))
    figure.update_layout(height=620, margin=dict(l=80, r=20, t=30, b=70),
        paper_bgcolor="white", plot_bgcolor="white",
        font=dict(family="Arial, sans-serif", color="#17334b"),
        xaxis=dict(title="Trial: year × location × management", tickmode="array",
            tickvals=[i for i, t in enumerate(trials) if i == 0 or t[0] != trials[i-1][0]],
            ticktext=[str(t[0]) for i, t in enumerate(trials) if i == 0 or t[0] != trials[i-1][0]]),
        yaxis=dict(title="Cultivars (training, validation, test)" if complete else "Cultivars (ordered by last subset year)",
                   showticklabels=False, autorange="reversed"))
    total = len(genotype_order) * len(trials)
    missing = total - len(lookup)
    legend = (("Train", "#11877f"), ("Validation", "#e9a23b"), ("Test", "#e05c71"),
              ("Not used for model", "#c7d9e7")) if subsetted else (
                  ("Observed", "#11877f"),)
    counts = {label: sum(
        ("Not used for model" if splitter(r) == "Not used" else splitter(r)) == label for r in rows)
        if subsetted else len(rows) for label, _ in legend}
    summary = (
        f"<p><b>{len(genotype_order)} cultivars × {len(trials)} trials</b> · "
        f"{len(lookup):,} observed means · {sum(r['plots'] for r in rows):,} contributing plots.<br>"
        f"{missing:,} ({100 * missing / total:.1f}%) cells "
        + ("unobserved or excluded by subsetting." if subsetted else "without recorded yield.")
        + "</p><p>" + " &nbsp; ".join(
            f'<span style="color:{color}">●</span> {label} {counts[label]:,}' for label, color in legend)
        + ' &nbsp; <span style="color:#aab8c2">●</span> '
        + ("Unobserved / excluded" if subsetted else "Missing yield") + "</p>")
    return figure, summary

landscape_toggle = widgets.ToggleButtons(
    options=[("Complete", "complete"), ("Sparse cohort", "sparse"), ("Original", "original")],
    value="complete", description="Dataset:")
initial_landscape, initial_summary = landscape_figure(True, complete=True)
landscape_plot = figure_widget(initial_landscape)
landscape_summary = widgets.HTML(initial_summary)

def update_landscape(change):
    figure, summary = landscape_figure(change["new"] != "original", complete=change["new"] == "complete")
    with landscape_plot.batch_update():
        landscape_plot.data[0].update(figure.data[0].to_plotly_json())
        plot_layout(landscape_plot).update(figure.layout.to_plotly_json())
    landscape_summary.value = summary

landscape_toggle.observe(update_landscape, names="value")
landscape_box = widgets.VBox([landscape_toggle, landscape_summary, landscape_plot])

#@title Load SNP features
import xml.etree.ElementTree as ET

multi_cache = {}
SNP_COUNT = 10_000

def load_snp_subset(training_ids=None):
    if training_ids is None and "genetic" in multi_cache:
        return multi_cache["genetic"]
    if debug_dummy_snps:
        random = np.random.default_rng(42)
        matrix = random.integers(0, 3, size=(len(snp_ids), SNP_COUNT)).astype(np.float32)
        genetic = {ident: matrix[i] for i, ident in enumerate(snp_ids)}
        if training_ids is None:
            multi_cache["eligible_markers"] = SNP_COUNT
            multi_cache["genetic"] = genetic
            multi_cache["markers"] = [f"DEBUG_SNP_{i + 1}" for i in range(SNP_COUNT)]
        return genetic
    workbook_bytes = snp_workbook_bytes
    with zipfile.ZipFile(io.BytesIO(workbook_bytes)) as workbook:
        namespace = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
        strings = []
        with workbook.open("xl/sharedStrings.xml") as source:
            for _, element in ET.iterparse(source, events=("end",)):
                if element.tag == namespace + "si":
                    strings.append("".join(node.text or "" for node in element.iter(namespace + "t")))
                    element.clear()

        with zipfile.ZipFile(io.BytesIO(archive_bytes)) as trial_archive:
            with trial_archive.open("Briwecs_data/metadata/BRIWECs_cultivar_info.csv") as raw:
                metadata = {int(row["BRISONr"].split("_")[1]): row["genotype"].strip()
                            for row in csv.DictReader(io.TextIOWrapper(raw, encoding="cp1252"))}
        known_aliases = {
            28: ("Robigous", "Robigus"), 79: ("Brillant", "Brilliant"),
            140: ("Capelle Desprez", "Cappelle Desprez"), 183: ("Boregan", "Boregar"),
            187: ('Triple dirk "S"', 'Triple Dirk "S"'), 199: ("Mex. 3", "Mexico 3"),
            208: ("WW 4180 (Kongo)", "WW 4180"), 212: ("Lambriego Inia", "Labriego-Inia"),
        }
        ids = names = None
        selected = []
        eligible_count = 0
        marker_rng = np.random.default_rng(42)
        train_columns = []
        print("Reading SNP workbook; this can take several minutes...", flush=True)
        with workbook.open("xl/worksheets/sheet1.xml") as source:
            for _, row in ET.iterparse(source, events=("end",)):
                if row.tag != namespace + "row":
                    continue
                number = int(row.attrib["r"])
                cells = list(row)
                if number == 1:
                    ids = [int(c.find(namespace + "v").text) for c in cells[11:]]
                    if len(ids) != 221 or len(set(ids)) != len(ids):
                        raise ValueError("Unexpected or duplicate SNP sample identifiers.")
                    observed_training_ids = (set(training_ids) if training_ids is not None
                                             else {r["genotype"] for r in observations
                                                   if split_name(r) == "Train"})
                    train_columns = [i for i, sample in enumerate(ids)
                                     if f"BRISONr_{sample}" in observed_training_ids]
                elif number == 2:
                    names = [strings[int(c.find(namespace + "v").text)].strip() for c in cells[11:]]
                    mismatches = [(sample, name, metadata.get(sample))
                                  for sample, name in zip(ids, names)
                                  if metadata.get(sample) != name and
                                  known_aliases.get(sample) != (name, metadata.get(sample))]
                    if mismatches:
                        raise ValueError(f"SNP sample names do not match cultivar metadata: {mismatches[:5]}")
                else:
                    if len(cells) != 232:
                        raise ValueError(f"Unexpected SNP workbook row shape at row {number}.")
                    marker = strings[int(cells[0].find(namespace + "v").text)]
                    calls = [strings[int(c.find(namespace + "v").text)]
                             for c in cells[11:]]
                    alleles = sorted({base for i in train_columns for base in calls[i]
                                      if base in "ACGT"})
                    if len(alleles) == 2:
                        def dosage(call):
                            if len(call) != 2 or any(base not in alleles for base in call):
                                return np.nan
                            return float(call.count(alleles[1]))
                        train_values = np.asarray([dosage(calls[i]) for i in train_columns])
                        valid = train_values[np.isfinite(train_values)]
                        if len(valid) >= 0.9 * len(train_columns) and np.std(valid) > 0.1:
                            eligible_count += 1
                            slot = (eligible_count - 1 if eligible_count <= SNP_COUNT
                                    else int(marker_rng.integers(eligible_count)))
                            if slot < SNP_COUNT:
                                values = np.asarray([dosage(call) for call in calls], dtype=np.float32)
                                if eligible_count <= SNP_COUNT:
                                    selected.append((marker, values))
                                else:
                                    selected[slot] = (marker, values)
                row.clear()
    if len(selected) < SNP_COUNT:
        raise ValueError(f"Found only {len(selected):,} qualifying SNPs; need {SNP_COUNT:,}.")
    marker_matrix = np.stack([values for _, values in selected], axis=1)
    genetic = {f"BRISONr_{sample}": marker_matrix[i] for i, sample in enumerate(ids)}
    if training_ids is None:
        multi_cache["eligible_markers"] = eligible_count
        multi_cache["genetic"] = genetic
        multi_cache["markers"] = [marker for marker, _ in selected]
    return genetic

def snp_training_modes(training_calls):
    calls = np.asarray(training_calls)
    if calls.ndim != 2 or not len(calls):
        raise ValueError("Training SNP calls must be a non-empty matrix.")
    observed = calls[np.isfinite(calls)]
    if not np.isin(observed, (0, 1, 2)).all():
        raise ValueError("SNP dosages must be 0, 1, 2, or missing.")
    counts = np.stack([(calls == dosage).sum(axis=0) for dosage in (0, 1, 2)])
    if np.any(counts.sum(axis=0) == 0):
        raise ValueError("A selected SNP has no observed training calls.")
    return np.argmax(counts, axis=0).astype(np.float32)


load_snp_subset()
print("SNP features are ready.", flush=True)


#@title Prepare weather, soil and management inputs
import re
import warnings
import openpyxl

def load_soil():
    soil = {}
    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
        names = [name for name in archive.namelist()
                 if re.search(r"/Management_information_[A-Z]+_\d{4}\.xlsx$", name)]
        for name in names:
            match = re.search(r"Management_information_([A-Z]+)_(\d{4})\.xlsx$", name)
            key = (match.group(1), int(match.group(2)))
            if key in soil:
                raise ValueError(f"Duplicate management workbook for {key}")
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore", message="Workbook contains no default style")
                wb = openpyxl.load_workbook(io.BytesIO(archive.read(name)),
                                           read_only=True, data_only=True)
            fields = {}
            for row in wb.worksheets[0].iter_rows(values_only=True):
                if row and row[0] is not None:
                    fields[str(row[0]).strip().lower()] = row[1] if len(row) > 1 else None
            wb.close()
            def number(label, low, high):
                value = fields.get(label)
                if isinstance(value, (int, float)) and not isinstance(value, bool):
                    return float(value) if low <= value <= high else np.nan
                return np.nan
            kind = fields.get("type of soil")
            kind = str(kind).strip().lower() if kind is not None else ""
            if kind in ("", "?", "none", "nan"):
                kind = "__missing__"
            soil[key] = {"soil type": kind,
                         "soil productivity": number("grade of soil productivity (ackerzahl)", 0, 100),
                         "soil pH": number("ph", 0, 14),
                         "clay fraction": number("clay", 0, 1),
                         "sand fraction": number("sand", 0, 1),
                         "silt fraction": number("silt", 0, 1)}
    if not soil:
        raise ValueError("No management workbooks found for soil features.")
    return soil

soil_by_site_year = load_soil()

def environment_features(rows, train_mask):
    numeric = list(weather_columns) + ["soil productivity", "soil pH", "clay fraction",
                                      "sand fraction", "silt fraction"]
    values, categories = [], []
    for row in rows:
        soil = soil_by_site_year.get((row["location"], row["year"]), {})
        values.append([row[name] if name in weather_columns else soil.get(name, np.nan)
                       for name in numeric])
        parts = row["management"].split("_")
        if len(parts) != 3:
            raise ValueError(f"Unexpected management code: {row['management']}")
        categories.append(parts + [soil.get("soil type", "__missing__")])
    values = np.asarray(values, dtype=float)
    missing = ~np.isfinite(values)
    means = np.nanmean(values[train_mask], axis=0)
    if not np.all(np.isfinite(means)):
        raise ValueError("A soil or weather feature has no training observations.")
    filled = np.where(missing, means, values)
    scale = filled[train_mask].std(axis=0)
    scale[scale < 1e-6] = 1.0
    encoded = [(filled - means) / scale, missing.astype(float)]
    categories = np.asarray(categories, dtype=object)
    for col in range(categories.shape[1]):
        known = sorted(set(categories[train_mask, col]) - {"__missing__"})
        encoded.append(np.asarray([[float(value == item) for item in known] +
                                   [float(value == "__missing__"),
                                    float(value != "__missing__" and value not in known)]
                                   for value in categories[:, col]], dtype=float))
    return np.concatenate(encoded, axis=1).astype(np.float32)

#@title Explore the BRIWECS dataset
# These views describe the dataset; the descriptive PCA is not a model input.
from scipy.stats import gaussian_kde

dataset_citation = (
    '<a href="https://doi.org/10.1038/s41597-024-04332-7" target="_blank">'
    'Wang et al. (2025). Multi-environment field trials for wheat yield, stability '
    'and breeding progress in Germany. <i>Scientific Data</i> 12, 64.</a>')
snp_citation = (
    '<a href="https://doi.org/10.3389/fpls.2019.01771" target="_blank">'
    'Lichthardt et al. (2020), <i>Frontiers in Plant Science</i> 10:1771</a>')
card_style = """
<style>
.briwecs-card{font-family:Arial,sans-serif;color:#17334b;padding:22px 24px;
 background:linear-gradient(120deg,#f2f8f7,#f8fafc);border:1px solid #dbe6e9;
 border-radius:14px;margin:12px 0;line-height:1.6}
.briwecs-card h2{font-size:23px;margin:0 0 8px}.briwecs-card p{margin:8px 0}
.briwecs-card a{color:#087e78}.briwecs-citation{background:white;padding:14px 18px;
 border-left:4px solid #11877f;border-radius:6px;margin:12px 0}
.briwecs-stats{display:flex;flex-wrap:wrap;gap:12px;margin:18px 0}
.briwecs-stat{background:white;border:1px solid #dbe6e9;border-radius:10px;
 padding:12px 18px;min-width:110px}.briwecs-stat b{display:block;font-size:25px;color:#11877f}
.briwecs-stat span{font-size:12px;color:#536b7b}
.briwecs-table{border-collapse:collapse;width:100%;background:white;font-size:13px}
.briwecs-table th{text-align:left;background:#e8f2f1;padding:10px}
.briwecs-table td{padding:9px 10px;border-bottom:1px solid #e6edef}
.briwecs-note{font-size:13px;color:#536b7b}
</style>
"""

def info_card(title, body):
    return widgets.HTML(f'<section class="briwecs-card"><h2>{title}</h2>{body}</section>')

def info_table(headers, rows):
    return ('<table class="briwecs-table"><thead><tr>'
            + "".join(f"<th>{html.escape(str(h))}</th>" for h in headers)
            + "</tr></thead><tbody>"
            + "".join("<tr>" + "".join(f"<td>{html.escape(str(v))}</td>" for v in row) + "</tr>" for row in rows)
            + "</tbody></table>")

def style_dataset_figure(figure, height=440):
    figure.update_layout(height=height, font=dict(family="Arial, sans-serif", color="#17334b"),
        paper_bgcolor="white", plot_bgcolor="white", margin=dict(l=65, r=35, t=65, b=65),
        legend=dict(orientation="h", y=1.13, x=0))
    figure.update_xaxes(gridcolor="#e9eff2", zeroline=False)
    figure.update_yaxes(gridcolor="#e9eff2", zeroline=False)
    return figure

original_ids = {r["genotype"] for r in exploration_observations}
metadata_by_id = {r["BRISONr"]: r for r in cultivar_info}
original_years = sorted({r["year"] for r in exploration_observations})
locations_years = sorted({(r["year"], r["location"]) for r in exploration_observations})
managements = sorted({r["management"] for r in exploration_observations})
statistics = "".join(
    f'<div class="briwecs-stat"><b>{value:,}</b><span>{label}</span></div>'
    for value, label in ((len(original_ids), "cultivars with yield"), (len(site_names), "German sites"),
                         (len(original_years), "harvest years"), (len(trials), "recorded yield trials"),
                         (len(exploration_subset), "subset cultivar × trial means")))
description = info_card("A multi-environment record of wheat breeding",
    f'<div class="briwecs-citation">{dataset_citation}<br>'
    '<a href="https://doi.org/10.6084/m9.figshare.27910269.v1" target="_blank">'
    'Original dataset · Figshare · CC BY 4.0</a></div>'
    '<p>BRIWECS evaluates winter wheat (<i>Triticum aestivum</i> L.) across six German '
    'locations and harvest years 2015–2020. The published resource comprises 228 cultivars '
    'released between 1963 and 2016, 24 traits and 526,751 trait observations. '
    'It supports analysis of genotype × environment × management interactions, yield stability '
    'and historical breeding progress.</p>'
    f'<div class="briwecs-stats">{statistics}</div>'
    '<p>The original experiments were replicated field trials; the location × year × '
    'management combinations are unbalanced. The notebook averages available plot yields '
    'within each cultivar × trial rather than using the spatially adjusted BLUEs supplied '
    'with the publication.</p>')

with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    with archive.open("Briwecs_data/metadata/BRIWECS_location_coordinates.csv") as raw:
        coordinates = list(csv.DictReader(io.TextIOWrapper(raw, encoding="utf-8-sig")))
site_colors = dict(zip(site_names, ("#11877f", "#407fa5", "#bd7844", "#7864a8", "#ce6678", "#7d913c")))
site_map = go.Figure()
for point in coordinates:
    site = point["Location"]
    site_rows = [r for r in exploration_observations if r["location"] == site]
    site_years = sorted({r["year"] for r in site_rows})
    site_trace = go.Scattergeo if in_colab else go.Scattermap
    site_map.add_trace(site_trace(
        lon=[float(point["Long"])], lat=[float(point["Lat"])],
        mode="markers+text", text=[site_names[site]], textposition="top right",
        textfont=dict(size=13, color="#17334b"), name=site_names[site],
        marker=dict(size=14, color=site_colors[site]), showlegend=False,
        customdata=[[f"{site_years[0]}–{site_years[-1]}", len({r["management"] for r in site_rows}), len(site_rows)]],
        hovertemplate=f"<b>{site_names[site]}</b><br>"
            "%{customdata[0]}<br>%{customdata[1]} management classes"
            "<br>%{customdata[2]:,} cultivar × trial means<extra></extra>"))
if in_colab:
    site_map.update_geos(scope="europe", projection_type="mercator",
        lonaxis_range=[5, 16], lataxis_range=[47, 56], showland=True, landcolor="#eef3ef",
        showcountries=True, countrycolor="#aab8c2", showocean=True, oceancolor="#e4eff5")
else:
    site_map.update_layout(map=dict(style="carto-positron", center=dict(lat=51.7, lon=9.8), zoom=4.9))
site_map.update_layout(height=530, margin=dict(l=0, r=0, t=0, b=0), paper_bgcolor="white",
    font=dict(family="Arial, sans-serif", color="#17334b"))
site_map = ColabPlot(site_map)
site_details = info_table(("Location", "Code", "Harvest years", "Yield trials"),
    [(site_names[site], site, ", ".join(str(y) for y in sorted({r["year"] for r in exploration_observations if r["location"] == site})),
      len({(r["year"], r["management"]) for r in exploration_observations if r["location"] == site}))
     for site in site_names])
locations_panel = widgets.VBox([info_card("Six locations, contrasting growing environments",
    '<p>The field network spans northern and central Germany, combining variation in climate '
    'and soil with site-specific management contrasts. Coordinates are taken directly from '
    f'the deposited metadata; site names follow {dataset_citation}.</p>'
    '<p class="briwecs-note">Pan and zoom to explore the sites; hover over a marker for coverage. '
    + ('Colab uses an SVG country-outline map to avoid WebGL/tile-rendering failures. '
       'Internet is required for country boundaries.</p>' if in_colab else
       'Basemap: CARTO / OpenStreetMap; an internet connection is required for map tiles.</p>')),
    site_map, info_card("Field-network coverage", site_details)])

landscape_panel = widgets.VBox([info_card("Design overview · cultivar × trial coverage",
    '<p><b>Complete</b> uses all available 2015-2017 records of randomly assigned training genotypes, '
    'validation genotypes in 2018, and test genotypes in 2019 (Sections 01-03). '
    '<b>Sparse cohort</b> uses simulated breeding cohorts (Sections 04-06), including '
    'retained follow-up records; only designated training and holdout records enter model scoring. '
    '<b>Original</b> shows all recorded combinations. Columns are trials (year × location × management); '
    'rows are cultivars grouped as training, validation, then test for Complete (sorted by identifier within each group), '
    'and ordered by last subset year for Sparse cohort.</p>'
    '<p>Colors identify training, validation, test and records not used for models; gray means '
    'missing or excluded. Test cultivars appear here and in SNP PCA without yield values; '
    'other exploratory views omit test genotypes from both dataset splits.</p>'), landscape_box])

ids = sorted(multi_cache["genetic"])
calls = np.stack([multi_cache["genetic"][ident] for ident in ids])
filled = np.where(np.isfinite(calls), calls, snp_training_modes(calls))
centered = filled - filled.mean(axis=0)
sd = centered.std(axis=0)
sd[sd < 1e-6] = 1.
pca_device = torch.device("cpu") if device.type == "mps" else device
standard = torch.as_tensor((centered / sd).astype(np.float32), device=pca_device)
gram = standard @ standard.T
values, vectors = torch.linalg.eigh(gram)
order = torch.argsort(values, descending=True)[:3]
scores = (vectors[:, order] * torch.sqrt(torch.clamp(values[order], min=0))).cpu().numpy()
explained = (torch.clamp(values[order], min=0) / torch.clamp(values, min=0).sum()).cpu().numpy()

def pca_split(ident):
    return ("Train" if ident in train_genotypes else "Validation" if ident in val_genotypes
            else "Test" if ident in test_genotypes else "Not assigned")

pca = figure_widget()
for group, color in (*split_colors.items(), ("Not assigned", "#aab8c2")):
    indices = [i for i, ident in enumerate(ids) if pca_split(ident) == group]
    if indices:
        details = []
        for i in indices:
            ident = ids[i]
            meta = metadata_by_id.get(ident, {})
            details.append([meta.get("genotype", ident), ident, meta.get("RYear", "NA"),
                            meta.get("country", "NA"), meta.get("baking_qulaity", "NA"),
                            pca_split(ident)])
        pca.add_scatter3d(x=scores[indices, 0], y=scores[indices, 1], z=scores[indices, 2],
            mode="markers", name=group,
            customdata=details, marker=dict(color=color, size=5, opacity=.8, line=dict(color="white", width=.7)),
            hovertemplate="<b>%{customdata[0]}</b> · %{customdata[1]}<br>Release year: %{customdata[2]}"
                "<br>Country: %{customdata[3]} · Baking class: %{customdata[4]}"
                "<br>Sparse cohort split: %{customdata[5]}<extra>%{fullData.name}</extra>")
style_dataset_figure(pca, 560)
pca.update_layout(title=dict(text="Genetic relationships (colors: sparse cohort split)", y=0.98, yanchor="top"),
    legend=dict(orientation="h", x=0, y=1, yanchor="bottom"),
    scene=dict(xaxis=dict(title=f"PC1 ({100 * explained[0]:.1f}% variance)"),
               yaxis=dict(title=f"PC2 ({100 * explained[1]:.1f}% variance)"),
               zaxis=dict(title=f"PC3 ({100 * explained[2]:.1f}% variance)"),
               aspectmode="data"), margin=dict(l=0, r=0, t=110, b=0))
genotypes_panel = widgets.VBox([info_card("Genetic diversity and breeding history",
    '<p>The BRIWECS panel spans historical and modern winter-wheat cultivars. '
    f'The associated SNP study, {snp_citation}, describes a German breeding-history collection '
    'of 174 cultivars (169 pure lines and five hybrids) and 46 diverse seed-bank accessions. '
    'The breeding-history collection covers baking-quality classes E, A, B and C.</p>'
    + ('<p><b>Debug mode: synthetic SNPs.</b> SNP PCA and model scores are for debugging only.</p>'
       if debug_dummy_snps else
    f'<p>The supplementary workbook contains {len(ids)} genotyped samples. This notebook uses '
    f'<b>{len(multi_cache["markers"]):,} SNPs</b>, selected reproducibly from '
    f'{multi_cache["eligible_markers"]:,} markers meeting training-panel biallelic, '
    'call-rate (≥90%) and dosage-variation (standard deviation >0.1) criteria.</p>')
    + '<p>The three-dimensional PCA displays the first three principal components, with '
    'axis labels reporting explained variance. Colors distinguish training (teal), '
    'validation (amber) and test (rose) cultivars; samples outside the model cohorts are gray. '
    'Drag to rotate and scroll to zoom. Hover labels include baking class, release year '
    'and model split. These split assignments are not inferred genetic clusters.</p>'
    '<p class="briwecs-note">PCA uses mode-imputed, centered and standardized SNP dosages for '
    'all genotyped samples, including test cultivars, without phenotype values. This descriptive ordination is not '
    'a model input; predictive preprocessing remains training-only.</p>'), pca])
del calls, filled, centered, standard, gram

management_labels = {
    "HN": "High nitrogen (220 kg N/ha)", "LN": "Low nitrogen (110 kg N/ha)",
    "WF": "With fungicide", "NF": "Without fungicide",
    "RF": "Rain-fed", "IR": "Irrigated", "RO": "Rain-out shelter",
}

def management_description(code):
    parts = code.split("_")
    if len(parts) != 3 or parts[0] not in ("HN", "LN") or parts[1] not in ("WF", "NF") or parts[2] not in ("RF", "IR", "RO"):
        raise ValueError(f"Unexpected BRIWECS management code: {code}")
    return " · ".join(management_labels[p] for p in parts)

available = {(r["year"], r["location"], r["management"]) for r in exploration_observations}
subset_available = {(r["year"], r["location"], r["management"]) for r in exploration_subset}
grid_values = [[2 if (year, location, m) in subset_available else 1 if (year, location, m) in available else 0
                for m in managements] for year, location in locations_years]
grid_hover = [[f"{year} · {site_names[location]}<br>{m}<br>{management_description(m)}<br>"
               + ("Recorded; retained in subset" if value == 2 else "Recorded; original only" if value == 1 else "No recorded yield trial")
               for m, value in zip(managements, row)] for (year, location), row in zip(locations_years, grid_values)]
grid = figure_widget(go.Heatmap(z=grid_values, x=managements,
    y=[f"{year} · {site_names[location]}" for year, location in locations_years], customdata=grid_hover,
    zmin=0, zmax=2, xgap=3, ygap=3, showscale=False,
    colorscale=[[0, "#eef1f3"], [.2499, "#eef1f3"], [.25, "#aac8d9"], [.7499, "#aac8d9"],
                [.75, "#11877f"], [1, "#11877f"]], hovertemplate="%{customdata}<extra></extra>"))
style_dataset_figure(grid, 850)
grid.update_layout(title="Environment × management coverage", margin=dict(l=180, r=20, t=60, b=110),
    xaxis_title="Management class", yaxis_title="Environment: harvest year × location",
    yaxis=dict(autorange="reversed"), xaxis=dict(tickangle=-35))
management_table = info_table(("Code", "Management class"), [(code, management_description(code)) for code in managements])
weather_reductions = {"sum": "monthly total", "mean": "monthly mean",
                      "min": "monthly minimum", "max": "monthly maximum"}
weather_list = "<ul>" + "".join(
    f"<li>{html.escape(label)} — {weather_reductions[reduction]}</li>"
    for label, (_, reduction) in weather_variables.items()) + "</ul>"
trials_panel = widgets.VBox([info_card("Environments and management contrasts",
    f'<p>An <b>environment (E)</b> is a location × harvest-year combination; a <b>trial (E×M)</b> '
    f'adds its management class. The yield records cover {len(locations_years)} environments '
    f'and {len(managements)} management classes, producing {len(trials)} trials. '
    'The experimental design was predominantly randomized blocks in 2015–2017 and '
    'treatment-factorial in 2018–2020, with site-specific exceptions reported in the paper.</p>'
    '<p>Management classes combine nitrogen supply, fungicide protection and water availability. '
    'The published nitrogen levels are HN = 220 and LN = 110 kg N ha⁻¹; WF/NF denotes '
    'with/without fungicide, and RF/IR/RO denotes rain-fed, irrigated and rain-out-shelter '
    f'conditions, respectively ({dataset_citation}, Fig. 2). Not every treatment combination '
    'was implemented at every site and year.</p>'
    + management_table
    + f'<p>Model environmental covariates comprise {len(weather_columns)} monthly weather summaries '
    '(seven variables over nine months from median sowing: precipitation, solar radiation, '
    'humidity, wind and mean/minimum/maximum air temperature), plus soil productivity, pH, '
    'clay, sand and silt fractions. Soil type and the three management factors are categorical '
    'inputs. Weather windows may extend beyond harvest.</p>'
    '<p class="briwecs-note">Grid: teal = recorded and retained in the subset; blue = recorded '
    'in the original only; gray = no recorded yield trial. Hover for full treatment definitions.</p>'), grid,
    info_card("Weather variables across the growing season",
        '<p>Daily DWD weather records are aggregated into nine monthly windows beginning '
        'at the trial’s median sowing date. Each variable supplies one covariate per window; '
        'these windows may extend beyond harvest.</p>' + weather_list)])

yield_years = original_years
all_yields = np.asarray([r["yield"] for r in exploration_observations])
yield_axis = np.linspace(min(0., float(all_yields.min())), float(all_yields.max()) + 5, 400)
yield_ridges = figure_widget(make_subplots(rows=1, cols=2, shared_xaxes=True, shared_yaxes=True,
    horizontal_spacing=.1, subplot_titles=("Sparse cohort dataset", "Original dataset")))
ridge_counts = {}
for column, (label, rows, color) in enumerate(
    (("Sparse cohort", exploration_subset, "#11877f"), ("Original", exploration_observations, "#407fa5")), start=1):
    for index, year in enumerate(yield_years):
        values_for_year = np.asarray([r["yield"] for r in rows if r["year"] == year])
        ridge_counts[(label, year)] = len(values_for_year)
        if len(values_for_year) >= 2 and np.ptp(values_for_year) > 0:
            density = gaussian_kde(values_for_year)(yield_axis)
            ridge = index + .78 * density / density.max()
            x = np.concatenate((yield_axis, yield_axis[::-1]))
            y = np.concatenate((ridge, np.full(len(yield_axis), index)))
            yield_ridges.add_scatter(x=x, y=y, mode="lines", fill="toself", line=dict(color=color, width=1.5),
                fillcolor="rgba(17,135,127,.28)" if column == 1 else "rgba(64,127,165,.28)",
                showlegend=False, name=f"{label} {year}",
                hovertemplate=f"{year} · {label}<br>{len(values_for_year):,} cultivar × trial means"
                    "<br>Yield: %{x:.1f} dt/ha<extra></extra>", row=1, col=column)
        elif len(values_for_year):
            yield_ridges.add_scatter(x=values_for_year, y=np.full(len(values_for_year), index), mode="markers",
                marker=dict(color=color, symbol="line-ns", size=14), showlegend=False,
                hovertemplate=f"{year} · {label}<br>Yield: %{{x:.1f}} dt/ha<extra></extra>", row=1, col=column)
        yield_ridges.add_annotation(xref="x domain" if column == 1 else "x2 domain",
            yref="y" if column == 1 else "y2", x=.99, y=index + .8,
            text=f"n = {len(values_for_year):,}", showarrow=False, xanchor="right",
            font=dict(size=11, color="#536b7b"))
style_dataset_figure(yield_ridges, 570)
yield_ridges.update_xaxes(title_text="Grain yield (dt/ha, dry mass)", range=[yield_axis[0], yield_axis[-1]])
yield_ridges.update_yaxes(tickmode="array", tickvals=list(range(len(yield_years))),
    ticktext=[str(y) for y in yield_years], range=[-.25, len(yield_years)], gridcolor="#edf2f4")
yield_ridges.update_yaxes(title_text="Harvest year", row=1, col=1)
trait_coverage = info_table(("Notebook trait", "Original means with value", "Sparse cohort means with value"),
    [(label, f"{sum(label in r['traits'] for r in exploration_observations):,}",
      f"{sum(label in r['traits'] for r in exploration_subset):,}") for label in trait_columns])
phenotype_panel = widgets.VBox([info_card("Phenotypes, with grain yield as the focal trait",
    '<p>The publication reports 24 traits covering agronomic performance, grain and biomass '
    'yield, baking quality, phenology and fungal disease severity. This notebook extracts six '
    'traits from records with an available grain-yield measurement.</p>'
    '<p>The focal response is whole-plot grain yield (<code>Seedyield</code>), expressed as '
    'dry mass in dt ha⁻¹ (1 dt ha⁻¹ = 0.1 t ha⁻¹). Replicates are averaged within cultivar × '
    'trial. Grain protein, thousand-grain weight, plant height, biomass and sedimentation '
    'are available as additional notebook traits.</p>'
    '<p>Ridgelines compare observed yield distributions by harvest year, with the subset '
    'on the left and the original data on the right. Both panels use identical yield and '
    'year axes. Each distribution is a Gaussian kernel density estimate with Scott’s bandwidth, '
    'scaled to equal peak height; labels report the number of contributing cultivar × trial means. '
    'Single-valued distributions are shown as marks rather than a density.</p>'
    '<p class="briwecs-note">These distributions are descriptive, not adjusted cultivar effects. '
    'Changes in cultivar composition, sites and management contribute to annual differences; '
    'the panels do not isolate genetic gain.</p>'), yield_ridges,
    info_card("Trait availability among yield-bearing means", trait_coverage)])

panels = [description, locations_panel, landscape_panel, genotypes_panel, trials_panel, phenotype_panel]
tabs = widgets.Tab(children=panels, layout=widgets.Layout(width="100%"))
for i, title in enumerate(("Description", "Locations", "Genotype × trial", "Genotypes",
                           "Trials (E×M)", "Phenotype")):
    tabs.set_title(i, title)
tabs.selected_index = 0

def refresh_dataset_tab(change):
    if change["new"] is not None:
        refresh_colab_plots(panels[change["new"]])

tabs.observe(refresh_dataset_tab, names="selected_index")
dataset_cards = widgets.VBox([widgets.HTML(card_style), tabs], layout=widgets.Layout(width="100%"))


Preparing interactive visuals for Colab...
Configured Colab's native widget manager.
Neural-network device: cuda (Tesla T4); GPU computation verified.
Ready: 34,355 yield records combined into 15642 cultivar × trial means.
Sparse cohort design: 209 cultivars; 3,379 model observations.
Complete design: 160 training, 34 validation, 34 test genotypes; 9,252 model observations.
Reading SNP workbook; this can take several minutes...
SNP features are ready.


In [ ]:
#@title Explore the BRIWECS dataset
tabs.selected_index = 0
display_panel(dataset_cards)


In [ ]:
#@title Prepare shared playground controls (no models are trained here)

training_playground_source = r'''
#@title Curve-fitting playground
import asyncio
from IPython import get_ipython
from matplotlib.colors import to_hex
from scipy.stats import rankdata

def build_model(neurons, layers, input_features=1):
    """Build a network mapping selected features to one yield prediction."""
    modules = []
    inputs = input_features
    for _ in range(layers):
        modules.append(nn.Linear(inputs, neurons))
        modules.append(nn.ReLU())
        inputs = neurons
    modules.append(nn.Linear(inputs, 1))
    return nn.Sequential(*modules)

label_style = {"description_width": "110px"}
slider_layout = widgets.Layout(width="350px", max_width="100%")
weather_features = {f"E: {name}": name for name in weather_columns}
management_features = {}
management_parts = ("Nitrogen", "Fungicide", "Water")
for index, part in enumerate(management_parts):
    levels = sorted({row["management"].split("_")[index] for row in observations
                     if split_name(row) == "Train"})
    for level in levels:
        management_features[f"M: Management: {part} = {level}"] = (index, level)
selected_snps = np.random.default_rng(19).choice(len(multi_cache["markers"]), size=10, replace=False)
snp_features = {f"G: Random example SNP {i}": int(index) for i, index in enumerate(selected_snps, 1)}
all_snps_label = "G: All loaded SNPs (10,000)"
all_env_label = "E: All ECs + management"
env_prefix = "E: All ECs + management: "
snp_pc_names = [f"G: SNP principal component {i}" for i in range(1, 5)]
ec_dropdown = widgets.Dropdown(
    options=list(weather_features) + list(management_features) + list(snp_features) + [all_snps_label, all_env_label],
    value="E: Month 4 after sowing: Rainfall (mm)", description="Feature",
    style=label_style, layout=widgets.Layout(width="560px", max_width="100%"))
add_feature = widgets.Button(description="Add", icon="plus")
remove_feature = widgets.Button(description="Remove", icon="minus")
default_model_inputs = ["E: Month 4 after sowing: Rainfall (mm)",
                        "E: Month 4 after sowing: Mean air temperature (°C)"]
feature_list = widgets.Select(options=default_model_inputs, value=default_model_inputs[0],
                              description="Model inputs", rows=4, style=label_style,
                              layout=widgets.Layout(width="560px", max_width="100%"))
axis_one = widgets.Dropdown(description="Plot feature", style=label_style,
                            layout=widgets.Layout(width="500px", max_width="100%"))
axis_two = widgets.Dropdown(description="Second feature", style=label_style,
                            layout=widgets.Layout(width="500px", max_width="100%"))
axis_status = widgets.HTML("Train a model to choose plot features.")
trait_dropdown = widgets.Dropdown(
    options=list(trait_columns), value=trait_label, description="Trait",
    style=label_style, layout=widgets.Layout(width="420px", max_width="100%"))
learning_rate = widgets.FloatLogSlider(
    value=0.001, base=10, min=-4, max=-1, step=0.1, description="Learning rate",
    readout_format=".4f", style=label_style, layout=slider_layout, continuous_update=False)
weight_decay = widgets.FloatLogSlider(
    value=0.0025118864315095794, base=10, min=-6, max=-1, step=0.2, description="Weight decay",
    readout_format=".5f", style=label_style, layout=slider_layout, continuous_update=False)
neurons = widgets.SelectionSlider(value=8, options=[2, 4, 8, 16, 32, 64, 128],
    description="Neurons / layer", style=label_style, layout=slider_layout, continuous_update=False)
layers = widgets.IntSlider(value=2, min=1, max=4, step=1,
    description="Hidden layers", style=label_style, layout=slider_layout, continuous_update=False)
epochs = widgets.IntSlider(value=100, min=50, max=600, step=25,
    description="Max epochs", style=label_style, layout=slider_layout, continuous_update=False)
batch_size = widgets.SelectionSlider(value=128, options=[8, 16, 32, 64, 128, 256, 512], description="Batch size",
    style=label_style, layout=slider_layout, continuous_update=False)
train_button = widgets.Button(description="Train / retrain", button_style="success",
    icon="play", layout=widgets.Layout(width="155px", height="38px"))
stop_button = widgets.Button(description="Stop training", button_style="warning",
    icon="stop", disabled=True, layout=widgets.Layout(width="155px", height="38px"))
training_state = {"task": None, "stop_requested": False, "mode": None}
progress = widgets.IntProgress(value=0, min=0, max=250, description="Progress",
    style={"bar_color": "#11877f"}, layout=widgets.Layout(width="300px"))
status = widgets.HTML()
network_box = widgets.VBox([widgets.HTML(
    '<div style="padding:24px;color:#587184">Train a model above to reveal its network.</div>')])
controls = [ec_dropdown, trait_dropdown, learning_rate, weight_decay,
            neurons, layers, epochs, batch_size, train_button, add_feature, remove_feature, feature_list,
            axis_one, axis_two]
trained_view = {}

def add_input(_):
    if ec_dropdown.value in feature_list.options:
        status.value = "<b>Feature already added.</b>"
        return
    feature_list.options = list(feature_list.options) + [ec_dropdown.value]
    feature_list.value = ec_dropdown.value
    status.value = "Feature added. Train again to use it."

def remove_input(_):
    if len(feature_list.options) == 1:
        status.value = "<b>Keep at least one model input.</b>"
        return
    options = [name for name in feature_list.options if name != feature_list.value]
    feature_list.options = options
    feature_list.value = options[0]
    status.value = "Feature removed. Train again to use the remaining inputs."

add_feature.on_click(add_input)
remove_feature.on_click(remove_input)

def make_dataset(features, trait, prediction_rows=None):
    needs_genetics = any(name in snp_features or name == all_snps_label for name in features)
    genetic = multi_cache["genetic"] if needs_genetics else {}
    if prediction_rows is None:
        source_rows = observations
    else:
        if any(split_name(row) != "Test" or trait not in row["traits"] for row in prediction_rows):
            raise ValueError("Prediction rows must be test-split rows with a placeholder target.")
        source_rows = [row for row in observations if split_name(row) != "Test"]
        source_rows = source_rows + list(prediction_rows)
    rows = [row for row in source_rows if trait in row["traits"]
            and split_name(row) != "Not used" and (not needs_genetics or row["genotype"] in genetic)]
    y = np.asarray([row["traits"][trait] for row in rows], dtype=np.float32)
    splits = np.asarray([split_name(row) for row in rows])
    masks = {name: splits == name for name in ("Train", "Validation", "Test")}
    for name, mask in masks.items():
        if not mask.any():
            raise ValueError(f"No {name.lower()} observations are available.")
    training_ids = sorted({row["genotype"] for row in rows if split_name(row) == "Train"})
    columns, feature_slices = [], {}
    env_names, env_discrete = [], set()
    offset = 0
    snp_plot = None
    for name in features:
        if name == all_env_label:
            env_names = list(weather_features) + list(management_features)
            env_columns = []
            for env_name in env_names:
                if env_name in management_features:
                    index, level = management_features[env_name]
                    env_columns.append([float(row["management"].split("_")[index] == level) for row in rows])
                else:
                    env_columns.append([row[weather_features[env_name]] for row in rows])
            soil_names = ["soil productivity", "soil pH", "clay fraction", "sand fraction", "silt fraction"]
            soils = [soil_by_site_year.get((row["location"], row["year"]), {}) for row in rows]
            for soil_name in soil_names:
                values = np.asarray([soil.get(soil_name, np.nan) for soil in soils], dtype=np.float32)
                missing = ~np.isfinite(values)
                mean = np.nanmean(values[masks["Train"]])
                if not np.isfinite(mean):
                    raise ValueError(f"No training data for {soil_name}.")
                env_columns.append(np.where(missing, mean, values))
                env_names.append(f"E: {soil_name}")
                env_columns.append(missing.astype(np.float32))
                env_names.append(f"E: {soil_name}: missing")
                env_discrete.add(f"E: {soil_name}: missing")
            categories = [soil.get("soil type", "__missing__") for soil in soils]
            known = sorted({categories[i] for i in np.flatnonzero(masks["Train"])} - {"__missing__"})
            for kind in known + ["__missing__", "__unseen__"]:
                env_names.append(f"E: Soil type: {kind}")
                env_discrete.add(f"E: Soil type: {kind}")
                env_columns.append([float(value == kind if kind != "__unseen__"
                                          else value != "__missing__" and value not in known)
                                    for value in categories])
            env_discrete.update(management_features)
            values = np.asarray(env_columns, dtype=np.float32).T
        elif name == all_snps_label:
            ids = sorted({row["genotype"] for row in rows})
            lookup = {ident: i for i, ident in enumerate(ids)}
            raw = np.stack([genetic[ident] for ident in ids])
            train_calls = np.stack([genetic[ident] for ident in training_ids])
            modes = snp_training_modes(train_calls)
            filled = np.where(np.isfinite(raw), raw, modes)
            train_filled = np.where(np.isfinite(train_calls), train_calls, modes)
            mean, scale = train_filled.mean(axis=0), train_filled.std(axis=0)
            scale[scale < 1e-6] = 1.0
            standardized = (filled - mean) / scale
            fit_ids = sorted({row["genotype"] for row in rows
                              if split_name(row) in ("Train", "Validation")})
            fit = standardized[[lookup[ident] for ident in fit_ids]]
            if len(fit_ids) < 5:
                raise ValueError("At least five training and validation cultivars are needed for four SNP PCs.")
            # MPS does not support symmetric eigendecomposition; use CPU for that backend.
            pca_device = torch.device("cpu") if device.type == "mps" else device
            fit_tensor = torch.as_tensor(fit, device=pca_device)
            center_tensor = fit_tensor.mean(dim=0)
            centered = fit_tensor - center_tensor
            eigenvalues, eigenvectors = torch.linalg.eigh(centered @ centered.T)
            eigenvalues, eigenvectors = eigenvalues[-4:].flip(0), eigenvectors[:, -4:].flip(1)
            if bool(torch.any(eigenvalues <= 1e-8)):
                raise ValueError("The training and validation SNP matrix has fewer than four varying components.")
            components_tensor = (eigenvectors.T @ centered) / torch.sqrt(eigenvalues[:, None])
            signs = torch.sign(components_tensor.gather(1, components_tensor.abs().argmax(dim=1, keepdim=True)))
            components_tensor = components_tensor * signs
            scores_tensor = (torch.as_tensor(standardized, device=pca_device) - center_tensor) @ components_tensor.T
            center = center_tensor.cpu().numpy()
            components = components_tensor.cpu().numpy()
            scores = scores_tensor.cpu().numpy()
            row_indices = np.asarray([lookup[row["genotype"]] for row in rows])
            values = filled[row_indices]
            snp_plot = {"center": center, "components": components, "scores": scores[row_indices],
                        "mean": mean, "scale": scale}
        elif name in snp_features:
            marker = snp_features[name]
            calls = np.asarray([genetic[ident][marker] for ident in training_ids])
            if not np.isfinite(calls).any():
                raise ValueError(f"{name} has no observed training calls.")
            mode = snp_training_modes(calls[:, None])[0]
            values = np.asarray([genetic[row["genotype"]][marker] for row in rows], dtype=np.float32)
            values = np.where(np.isfinite(values), values, mode)
        elif name in management_features:
            index, level = management_features[name]
            values = np.asarray([float(row["management"].split("_")[index] == level)
                                 for row in rows], dtype=np.float32)
        else:
            values = np.asarray([row[weather_features[name]] for row in rows], dtype=np.float32)
        if values.ndim == 1:
            values = values[:, None]
        columns.append(values)
        feature_slices[name] = slice(offset, offset + values.shape[1])
        offset += values.shape[1]
    x = np.concatenate(columns, axis=1).astype(np.float32, copy=False)
    xm, xs = x[masks["Train"]].mean(axis=0), x[masks["Train"]].std(axis=0)
    ym, ys = float(y[masks["Train"]].mean()), float(y[masks["Train"]].std())
    non_snp_indices = [feature_slices[name].start for name in features if name not in (all_snps_label, all_env_label)]
    if np.any(xs[non_snp_indices] <= 0) or ys <= 0:
        raise ValueError("A training feature or yield has no variation; choose different inputs.")
    xs[xs < 1e-6] = 1.0
    plot_columns = {name: feature_slices[name].start for name in features
                    if name not in (all_snps_label, all_env_label)}
    if all_env_label in features:
        start = feature_slices[all_env_label].start
        plot_columns.update({env_prefix + name: start + index
                             for index, name in enumerate(env_names)})
    return x, y, masks, (xm, xs, ym, ys), rows, {
        "slices": feature_slices, "snp": snp_plot, "plot_columns": plot_columns,
        "env_discrete": env_discrete}



def safe_correlation(a, b):
    a, b = np.asarray(a, dtype=np.float64), np.asarray(b, dtype=np.float64)
    if not np.all(np.isfinite(a)) or not np.all(np.isfinite(b)):
        raise ValueError("Cannot calculate correlation with non-finite values.")
    if len(a) < 2 or np.std(a) < 1e-8 or np.std(b) < 1e-8:
        return None
    return float(np.corrcoef(a, b)[0, 1])


def correlation_groups(rows):
    trial_groups, cultivar_groups = defaultdict(list), defaultdict(list)
    for i, row in enumerate(rows):
        trial_groups[(row["year"], row["location"], row["management"])].append(i)
        cultivar_groups[row["genotype"]].append(i)
    return list(trial_groups.values()), list(cultivar_groups.values())


def mean_group_correlation(groups, actual, predicted):
    values = [safe_correlation(actual[indices], predicted[indices]) for indices in groups]
    values = [value for value in values if value is not None]
    return (float(np.mean(values)), len(values)) if values else (None, 0)


def mean_group_rank_correlations(groups, actual, predicted):
    pearson, spearman = [], []
    for indices in groups:
        observed, estimate = actual[indices], predicted[indices]
        value = safe_correlation(observed, estimate)
        if value is not None:
            pearson.append(value)
        rank_value = safe_correlation(rankdata(observed, method="average"),
                                      rankdata(estimate, method="average"))
        if rank_value is not None:
            spearman.append(rank_value)
    return ((float(np.mean(pearson)), len(pearson)) if pearson else (None, 0),
            (float(np.mean(spearman)), len(spearman)) if spearman else (None, 0))


def grouped_ranking_metrics(rows, actual, predicted):
    trial_groups, genotype_groups = correlation_groups(rows)
    (genotype_pearson, trials), (genotype_spearman, _) = mean_group_rank_correlations(
        trial_groups, actual, predicted)
    environment_pearson, genotypes = mean_group_correlation(genotype_groups, actual, predicted)
    return {"genotype pearson": genotype_pearson, "genotype spearman": genotype_spearman,
            "trials": trials, "environment pearson": environment_pearson, "genotypes": genotypes}


def prediction_correlations(rows, actual, predicted):
    trial_groups, cultivar_groups = correlation_groups(rows)
    return (safe_correlation(actual, predicted),
            mean_group_correlation(trial_groups, actual, predicted),
            mean_group_correlation(cultivar_groups, actual, predicted))


def chart_layout(fig, height):
    fig.update_layout(
        height=height, margin=dict(l=60, r=25, t=75, b=65),
        paper_bgcolor="white", plot_bgcolor="#f7fafb",
        font=dict(family="Arial, sans-serif", color="#17334b", size=11),
        hoverlabel=dict(bgcolor="#17334b", font_size=12, font_color="white"),
        legend=dict(orientation="h", yanchor="bottom", y=1.03, xanchor="left", x=0),
    )
    fig.update_xaxes(showgrid=True, gridcolor="#e4edef", zeroline=False)
    fig.update_yaxes(showgrid=True, gridcolor="#e4edef", zeroline=False)


plot = figure_widget()
for name in ("Train", "Validation"):
    plot.add_scatter(x=[], y=[], mode="lines", name=f"{name} loss",
        line=dict(color=split_colors[name], width=3))
plot.add_scatter(x=[], y=[], mode="markers", name="Best checkpoint (restored)",
    marker=dict(color="#293e73", size=12, symbol="diamond", line=dict(color="white", width=1)),
    hovertemplate="Restored checkpoint<br>Epoch %{x}<br>Validation MSE: %{y:.5f}<extra></extra>")
chart_layout(plot, 380)
plot.update_layout(title="Loss curves · diamond marks the restored best-validation checkpoint")
plot.update_xaxes(title_text="Epoch")
plot.update_yaxes(title_text="Mean squared error · standardized")
feature_box = widgets.VBox([widgets.HTML("Train a model to see the feature plot.")])


prediction_plot = figure_widget()
for name in ("Train", "Validation"):
    prediction_plot.add_scatter(x=[], y=[], mode="markers", name=name,
        marker=dict(color=split_colors[name], size=7, opacity=0.55),
        hovertemplate="Actual: %{x:.1f}<br>Predicted: %{y:.1f}<extra>" + name + "</extra>")
prediction_plot.add_scatter(x=[], y=[], mode="lines", name="Perfect prediction",
                            line=dict(color="#293e73", dash="dash"))
chart_layout(prediction_plot, 430)
prediction_plot.update_layout(title="Predicted vs actual yield")
prediction_plot.update_xaxes(title_text="Actual yield (dt/ha)")
prediction_plot.update_yaxes(title_text="Predicted yield (dt/ha)")
prediction_plot.add_annotation(x=0.98, y=0.02, xref="paper", yref="paper",
                               xanchor="right", yanchor="bottom", showarrow=False,
                               align="left", text="RMSE · within-trial correlation (Pearson / Spearman)", bgcolor="white")

def feature_figure(model, data, features, shown, trait):
    x, y, masks, (xm, xs, ym, ys), rows, metadata = data
    slices, snp = metadata["slices"], metadata["snp"]
    def plot_values(name):
        if name in snp_pc_names:
            return snp["scores"][:, snp_pc_names.index(name)]
        return x[:, metadata["plot_columns"][name]]
    selected = masks["Train"] | masks["Validation"]
    typical = xm.copy()
    for name, index in metadata["plot_columns"].items():
        base_name = name[len(env_prefix):] if name.startswith(env_prefix) else name
        if base_name in management_features or base_name in snp_features or base_name in metadata["env_discrete"]:
            values, counts = np.unique(x[masks["Train"], index], return_counts=True)
            typical[index] = values[np.argmax(counts)]
    grids = []
    for name in shown:
        values = plot_values(name)[selected]
        base_name = name[len(env_prefix):] if name.startswith(env_prefix) else name
        if base_name in management_features or base_name in snp_features or base_name in metadata["env_discrete"]:
            grids.append(np.unique(values))
        else:
            grids.append(np.linspace(values.min(), values.max(), 30 if len(shown) == 2 else 250))
    mesh = np.meshgrid(*grids, indexing="xy")
    inputs = np.tile(typical, (mesh[0].size, 1))
    if snp is not None:
        snp_slice = slices[all_snps_label]
        base = (typical[snp_slice] - snp["mean"]) / snp["scale"]
        base_scores = (base - snp["center"]) @ snp["components"].T
    for name, values in zip(shown, mesh):
        if name in snp_pc_names:
            pc = snp_pc_names.index(name)
            delta = values.ravel() - base_scores[pc]
            inputs[:, snp_slice] += delta[:, None] * (snp["components"][pc] * snp["scale"])
        else:
            inputs[:, metadata["plot_columns"][name]] = values.ravel()
    with torch.no_grad():
        predicted = model(torch.from_numpy(((inputs - xm) / xs).astype(np.float32)).to(device)).cpu().numpy().ravel() * ys + ym
    figure = go.Figure()
    for name in ("Train", "Validation"):
        mask = masks[name]
        selected_rows = [row for row, keep in zip(rows, mask) if keep]
        custom = [[site_names[row["location"]], row["year"], row["management"], row["genotype"]]
                  for row in selected_rows]
        if len(shown) == 1:
            figure.add_scatter(x=plot_values(shown[0])[mask], y=y[mask], mode="markers",
                name=name, customdata=custom, marker=dict(color=split_colors[name], size=7, opacity=.6),
                hovertemplate="%{customdata[0]} %{customdata[1]}<br>%{customdata[3]}<br>Feature: %{x:.2f}<br>Observed: %{y:.1f}<extra>" + name + "</extra>")
        else:
            figure.add_scatter3d(x=plot_values(shown[0])[mask], y=plot_values(shown[1])[mask], z=y[mask],
                mode="markers", name=name, customdata=custom,
                marker=dict(color=split_colors[name], size=3, opacity=.65),
                hovertemplate="%{customdata[0]} %{customdata[1]}<br>%{customdata[3]}<br>Observed: %{z:.1f}<extra>" + name + "</extra>")
    discrete = any((name[len(env_prefix):] if name.startswith(env_prefix) else name)
                   in (set(management_features) | set(snp_features) | metadata["env_discrete"])
                   for name in shown)
    if len(shown) == 1:
        figure.add_scatter(x=mesh[0].ravel(), y=predicted, name="Model",
                           mode="markers" if discrete else "lines",
                           marker=dict(color="#293e73", size=9), line=dict(color="#293e73"))
        figure.update_xaxes(title_text=shown[0], title_font_size=11.2, tickfont_size=9.6)
        figure.update_yaxes(title_text=trait, title_font_size=11.2, tickfont_size=9.6)
    elif discrete:
        figure.add_scatter3d(x=mesh[0].ravel(), y=mesh[1].ravel(), z=predicted,
                             mode="markers", name="Model",
                             marker=dict(color="#293e73", size=3))
    else:
        figure.add_surface(x=mesh[0], y=mesh[1], z=predicted.reshape(mesh[0].shape),
                           name="Model", colorscale="Blues", opacity=.55, showscale=False)
    figure.update_layout(title="Features vs yield", height=520,
                         paper_bgcolor="white", plot_bgcolor="#f7fafb",
                         font=dict(family="Arial, sans-serif", color="#17334b"),
                         margin=dict(l=55, r=20, t=70, b=55),
                         legend=dict(orientation="h"))
    if len(shown) == 2:
        figure.update_layout(scene=dict(
            xaxis=dict(title=dict(text=shown[0], font=dict(size=11.2)), tickfont=dict(size=9.6)),
            yaxis=dict(title=dict(text=shown[1], font=dict(size=11.2)), tickfont=dict(size=9.6)),
            zaxis=dict(title=dict(text=trait, font=dict(size=11.2)), tickfont=dict(size=9.6))))
    return figure_widget(figure)


def update_feature_view(change=None):
    if not trained_view:
        return
    first, second = axis_one.value, axis_two.value
    if second == first:
        axis_status.value = "<b>Choose two different features.</b>"
        return
    shown = [first] + ([second] if second is not None else [])
    feature_box.children = [feature_figure(trained_view["model"], trained_view["data"],
                                           trained_view["features"], shown, trained_view["trait"])]
    if any(name in snp_pc_names for name in shown):
        axis_status.value = "SNP PCs show illustrative profiles, not real cultivars; the model uses all SNPs."
    else:
        axis_status.value = "Other inputs are held at typical training values."


axis_one.observe(update_feature_view, names="value")
axis_two.observe(update_feature_view, names="value")


def render_training(model, data, features, trait, history, completed, total, x_tensor, best_epoch=None):
    x, y, masks, (xm, xs, ym, ys), rows, _ = data
    selected = masks["Train"] | masks["Validation"]
    visible_y = y[selected]
    with torch.no_grad():
        fitted = model(x_tensor[torch.from_numpy(selected).to(device)]).cpu().numpy().ravel() * ys + ym
    with plot.batch_update():
        plot.data[0].x, plot.data[0].y = history["epoch"], history["train"]
        validation_epochs = history.get("validation_epoch", history["epoch"])
        validation_losses = history.get("validation_loss", history["validation"])
        plot.data[1].x, plot.data[1].y = validation_epochs, validation_losses
        if best_epoch is None and validation_losses:
            best_epoch = validation_epochs[int(np.argmin(validation_losses))]
        if best_epoch in validation_epochs:
            best_index = validation_epochs.index(best_epoch)
            plot.data[2].x, plot.data[2].y = [best_epoch], [validation_losses[best_index]]
        else:
            plot.data[2].x, plot.data[2].y = [], []
    with prediction_plot.batch_update():
        for i, name in enumerate(("Train", "Validation")):
            mask = masks[name][selected]
            actual, estimate = visible_y[mask], fitted[mask]
            actual_values = np.asarray(actual, dtype=float)
            estimate_values = np.asarray(estimate, dtype=float)
            prediction_plot.data[i].x = actual_values.tolist()
            prediction_plot.data[i].y = estimate_values.tolist()
        bounds = [float(min(visible_y.min(), fitted.min())),
                  float(max(visible_y.max(), fitted.max()))]
        prediction_plot.data[2].x = bounds
        prediction_plot.data[2].y = bounds
        plot_layout(prediction_plot).annotations[0].text = "Closer to the line is better"
    trained_view.update(model=model, data=data, features=features, trait=trait)
    if completed == total:
        update_feature_view()



def make_network_chart(model, train_x, scale, features, trait, metadata):
    xm, xs, ym, ys = scale
    values = [train_x.detach().cpu().numpy()]
    with torch.no_grad():
        current = train_x
        for module in model:
            current = module(current)
            if isinstance(module, (nn.ReLU, nn.Tanh, nn.Sigmoid)) or module is model[-1]:
                values.append(current.detach().cpu().numpy())
    magnitudes = [np.mean(np.abs(value), axis=0) for value in values]
    magnitudes[0] = np.asarray([float(np.mean(magnitudes[0][metadata["slices"][name]]))
                                for name in features])
    sizes = [len(value) for value in magnitudes]
    positions = [np.linspace(-1, 1, n).tolist() if n > 1 else [0.] for n in sizes]
    linear = [module for module in model if isinstance(module, nn.Linear)]
    figure = go.Figure()
    # Group lines by weight sign and strength to keep the graph responsive.
    for level, module in enumerate(linear):
        raw_weights = module.weight.detach().cpu().numpy()
        if level == 0:
            weights = np.column_stack([
                np.zeros(raw_weights.shape[0]) if name in (all_snps_label, all_env_label)
                else raw_weights[:, metadata["slices"][name].start] for name in features])
        else:
            weights = raw_weights
        maximum = max(float(np.max(np.abs(weights))), 1e-12)
        shown = np.ones_like(weights, dtype=bool)
        if weights.size > 2048:
            shown[:] = False
            strongest = np.argpartition(np.abs(weights), -4, axis=1)[:, -4:]
            shown[np.arange(weights.shape[0])[:, None], strongest] = True
        if level == 0:
            for bundle in (all_snps_label, all_env_label):
                if bundle not in features:
                    continue
                index = features.index(bundle)
                shown[:, index] = False
                left = positions[0][index]
                xx, yy = [], []
                for right_y in positions[1]:
                    xx.extend([0, 1, None])
                    yy.extend([left, right_y, None])
                figure.add_trace(go.Scatter(x=xx, y=yy, mode="lines",
                    line=dict(color="#9aa7ad", width=1.2), opacity=0.6,
                    hoverinfo="skip", showlegend=False))
        for sign, color in ((1, "#75b5ac"), (-1, "#c8a9bc")):
            for bucket, line_width in enumerate((0.6, 1.3, 2.4)):
                xx, yy = [], []
                for j, right_y in enumerate(positions[level + 1]):
                    for i, left_y in enumerate(positions[level]):
                        if not shown[j, i]:
                            continue
                        weight = float(weights[j, i])
                        category = min(2, int(abs(weight) / maximum * 3))
                        if (weight >= 0) == (sign == 1) and category == bucket:
                            xx.extend([level, level + 1, None])
                            yy.extend([left_y, right_y, None])
                if xx:
                    figure.add_trace(go.Scatter(x=xx, y=yy, mode="lines",
                        line=dict(color=color, width=line_width), opacity=0.5,
                        hoverinfo="skip", showlegend=False))
    for level, (layer_values, ypos) in enumerate(zip(magnitudes, positions)):
        maximum = max(float(np.max(layer_values)), 1e-12)
        colors = [to_hex(matplotlib.colormaps["YlGnBu"](0.18 + .75 * float(v) / maximum))
                  for v in layer_values]
        title = "Input" if level == 0 else ("Output" if level == len(sizes) - 1
                                              else f"Hidden layer {level}")
        details = []
        for i, magnitude in enumerate(layer_values):
            if level == 0:
                feature = features[i]
                column = metadata["slices"][feature]
                if feature in (all_snps_label, all_env_label):
                    details.append(f"<b>Input · {html.escape(feature)}</b>"
                        f"<br>{column.stop - column.start:,} separate model inputs, shown as one node."
                        "<br>Grey lines represent many weights; they have no single sign or strength."
                        f"<br>Average mean |activation|: {float(magnitude):.3f}")
                else:
                    index = column.start
                    details.append(f"<b>Input · {html.escape(feature)}</b><br>Batch: ({len(train_x)}, {train_x.shape[1]})"
                        f"<br>Standardize: (x − {xm[index]:.2f}) / {xs[index]:.2f}"
                        f"<br>Mean |activation|: {float(magnitude):.3f}")
            else:
                layer = linear[level - 1]
                activation = "Linear" if level == len(sizes) - 1 else type(model[2 * level - 1]).__name__
                formula = ("z = W·a + b; trait = z × training SD + training mean"
                           if level == len(sizes) - 1 else f"z = W·a + b; a = {activation}(z)")
                details.append(f"<b>{title} · neuron {i + 1}</b>"
                    f"<br>Input: ({len(train_x)}, {layer.in_features})"
                    f"<br>Output: ({len(train_x)}, {layer.out_features})"
                    f"<br>Weights: ({layer.out_features}, {layer.in_features}); bias: ({layer.out_features},)"
                    f"<br>Activation: {activation}<br>{formula}"
                    f"<br>Mean |activation|: {float(magnitude):.3f}")
        figure.add_trace(go.Scatter(
            x=[level] * len(ypos), y=ypos, mode="markers",
            marker=dict(size=min(23, max(5, 550 / max(sizes))), color=colors,
                        line=dict(color="#185b72", width=1)),
            text=details, hovertemplate="%{text}<extra></extra>", name=title, showlegend=False))
        subtitle = (f"{len(features)} features" if level == 0 else (trait if level == len(sizes)-1
                    else type(model[2 * level - 1]).__name__))
        figure.add_annotation(x=level, y=1.32, showarrow=False,
            text=f"<b>{html.escape(title)}</b><br><span style='font-size:10px'>{html.escape(subtitle)}</span>",
            font=dict(color="#17334b", size=13), xanchor="center")
    figure.update_layout(
        height=min(900, max(430, max(sizes) * 6)), margin=dict(l=70, r=70, t=50, b=45),
        paper_bgcolor="white", plot_bgcolor="#f7fafb",
        font=dict(family="Arial, sans-serif", color="#17334b"),
        hoverlabel=dict(bgcolor="#17334b", font_size=12, font_color="white"),
        xaxis=dict(visible=False, range=[-0.7, len(sizes) - 0.3]),
        yaxis=dict(visible=False, range=[-1.3, 1.6]),
    )
    return figure_widget(figure)


def validate_model(model, input_features):
    if not isinstance(model, nn.Sequential):
        raise ValueError("build_model must return nn.Sequential so the network can be visualized.")
    modules = list(model)
    if len(modules) < 3 or len(modules) % 2 != 1:
        raise ValueError("Use Linear + activation pairs for hidden layers and a final Linear output.")
    previous = input_features
    for index in range(0, len(modules) - 1, 2):
        linear, activation = modules[index:index + 2]
        if (not isinstance(linear, nn.Linear) or linear.in_features != previous
                or linear.bias is None or linear.out_features > 128
                or not isinstance(activation, (nn.ReLU, nn.Tanh, nn.Sigmoid))):
            raise ValueError("Each hidden layer must be a biased Linear layer with matching "
                             "input size and at most 128 neurons, followed by ReLU, Tanh or Sigmoid.")
        previous = linear.out_features
    output = modules[-1]
    if (not isinstance(output, nn.Linear) or output.in_features != previous
            or output.out_features != 1 or output.bias is None):
        raise ValueError("The final layer must be a biased Linear layer with one output.")
    if len(modules) > 13:
        raise ValueError("Use at most six hidden layers to keep the diagram readable.")


class EarlyStopping:
    def __init__(self, patience=1000):
        self.patience = patience
        self.best_loss = float("inf")
        self.best_epoch = 0
        self.best_params = None

    def update(self, model, loss, epoch):
        if not np.isfinite(loss):
            raise ValueError("Non-finite validation loss.")
        if loss < self.best_loss:
            self.best_loss = loss
            self.best_epoch = epoch
            self.best_params = [param.detach().clone() for param in model.parameters()]
        return epoch - self.best_epoch >= self.patience

    def restore(self, model):
        if self.best_params is None:
            raise RuntimeError("No validation checkpoint was recorded.")
        with torch.no_grad():
            for param, saved in zip(model.parameters(), self.best_params):
                param.copy_(saved)


first_history = []
run_sort = widgets.Dropdown(description="Sort by", options=[("Run number", "id"),
    ("Training plot RMSE", "Train rmse"), ("Training within-trial correlation", "Train pearson"),
    ("Training within-genotype correlation", "Train environment pearson"),
    ("Validation plot RMSE", "Validation rmse"), ("Validation within-trial correlation", "Validation pearson"),
    ("Validation within-genotype correlation", "Validation environment pearson")],
    value="id", layout=widgets.Layout(width="350px"))
run_descending = widgets.Checkbox(value=False, description="Descending")
run_table = widgets.HTML("<p>Train a model to populate this table.</p>")
run_restore_selector = widgets.Dropdown(options=[], description="Select run",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="430px", max_width="100%"))
run_restore_button = widgets.Button(description="Restore saved run", icon="history", disabled=True)
controls.extend([run_sort, run_descending, run_restore_selector, run_restore_button])

def run_metrics(model, data, trait, x_tensor, splits=("Train", "Validation")):
    x, y, masks, (xm, xs, ym, ys), rows, metadata = data
    metrics = {}
    for name in splits:
        mask = masks[name]
        selected_rows = [row for row, keep in zip(rows, mask) if keep]
        with torch.no_grad():
            predicted = model(x_tensor[torch.from_numpy(mask).to(device)]).cpu().numpy().ravel() * ys + ym
        errors = []
        for row, estimate in zip(selected_rows, predicted):
            key = (row["location"], row["year"], row["management"], row["genotype"])
            values = trait_groups[key][trait]
            if not values:
                raise ValueError(f"No plot-level measurements for {trait}: {key}.")
            errors.extend((np.asarray(values, dtype=float) - estimate) ** 2)
        if not np.isfinite(errors).all():
            raise ValueError(f"Non-finite plot-level errors for {name}.")
        trial_groups, genotype_groups = correlation_groups(selected_rows)
        pearson, count = mean_group_correlation(trial_groups, y[mask], predicted)
        environment_pearson, genotype_count = mean_group_correlation(genotype_groups, y[mask], predicted)
        metrics[name + " rmse"] = float(np.sqrt(np.mean(errors)))
        metrics[name + " pearson"] = pearson
        metrics[name + " trials"] = count
        metrics[name + " environment pearson"] = environment_pearson
        metrics[name + " genotypes"] = genotype_count
    return metrics

def refresh_first_history(change=None, select_latest=False):
    if "refresh_test_comparison" in globals():
        refresh_test_comparison()
    if not first_history:
        run_table.value = "<p>Train a model to populate this table.</p>"
        run_restore_selector.options = []
        run_restore_button.disabled = True
        if "refresh_posthoc_models" in globals():
            refresh_posthoc_models()
        return
    key = run_sort.value
    value = lambda item: item["id"] if key == "id" else item["metrics"][key]
    defined = [item for item in first_history if value(item) is not None]
    ordered = sorted(defined, key=lambda item: (value(item), item["id"]),
                     reverse=run_descending.value)
    ordered += [item for item in first_history if value(item) is None]

    def metric_cell(item, split, metric, count):
        score = item["metrics"][split + " " + metric]
        text = "undefined" if score is None else f"{score:.3f}"
        groups = item["metrics"][split + " " + count]
        return f"{text}<br><span style='color:#536b7b;font-size:12px'>{groups} groups</span>"

    rows = []
    for item in ordered:
        details = "<br>".join(html.escape(name) for name in item["features"])
        status = "<br><span style='color:#9a5c2c'>Stopped early</span>" if item["stopped"] else ""
        cells = [f"<b>Run {item['id']}</b>{status}",
                 f"<b>{html.escape(item['trait'])}</b><br>{details}"]
        for split in ("Train", "Validation"):
            cells.extend((f"{item['metrics'][split + ' rmse']:.3f}",
                          metric_cell(item, split, "pearson", "trials"),
                          metric_cell(item, split, "environment pearson", "genotypes")))
        rows.append("<tr>" + "".join(
            f"<td style='padding:9px 10px;border-bottom:1px solid #e6edef;vertical-align:top'>{cell}</td>"
            for cell in cells) + "</tr>")
    header_style = "padding:9px 10px;background:#e8f2f1;border-bottom:1px solid #d5e4e3;text-align:left"
    subheader = "<tr>" + "".join(
        f"<th style='{header_style}'>{label}</th>" for label in (
            "Plot-level RMSE (lower is better)", "Within-trial correlation (Pearson r)", "Within-genotype correlation (Pearson r)",
            "Plot-level RMSE (lower is better)", "Within-trial correlation (Pearson r)", "Within-genotype correlation (Pearson r)")) + "</tr>"
    group_header = (
        f"<tr><th rowspan='2' style='{header_style}'>Run</th>"
        f"<th rowspan='2' style='{header_style}'>Trait / model inputs</th>"
        f"<th colspan='3' style='{header_style}'>Training</th>"
        f"<th colspan='3' style='{header_style}'>Validation</th></tr>")
    run_table.value = (
        "<div style='width:100%;overflow-x:auto'>"
        "<table style='width:100%;min-width:1120px;border-collapse:collapse;"
        "font:13px Arial,sans-serif;color:#17334b;table-layout:fixed'>"
        "<colgroup><col style='width:7%'><col style='width:30%'>"
        "<col style='width:9%'><col style='width:12%'><col style='width:15%'>"
        "<col style='width:9%'><col style='width:12%'><col style='width:15%'></colgroup>"
        "<thead>" + group_header + subheader + "</thead><tbody>" + "".join(rows) + "</tbody></table></div>")

    previous = run_restore_selector.value
    options = [(f"Run {item['id']} - {item['trait']}", item["id"]) for item in ordered]
    run_restore_selector.options = options
    ids = {item["id"] for item in ordered}
    run_restore_selector.value = (first_history[-1]["id"] if select_latest or previous not in ids
                                 else previous)
    run_restore_button.disabled = False
    if "refresh_posthoc_models" in globals():
        refresh_posthoc_models(select_latest=select_latest)

def restore_first(run_id):
    item = next((entry for entry in first_history if entry["id"] == run_id), None)
    if item is None:
        raise ValueError("Select a saved run to restore.")
    features, trait = item["features"], item["trait"]
    feature_list.options = features
    feature_list.value = features[0]
    trait_dropdown.value = trait
    for control, value in zip([neurons,layers,epochs,batch_size,learning_rate,weight_decay], item["settings"]):
        control.value = value
    data = make_dataset(features,trait)
    x,y,masks,(xm,xs,ym,ys),rows,metadata = data
    x_tensor = torch.from_numpy(x.copy())
    x_tensor.sub_(torch.from_numpy(xm)).div_(torch.from_numpy(xs))
    x_tensor = x_tensor.to(device)
    model = build_model(neurons.value,layers.value,x.shape[1]).to(device)
    validate_model(model,x.shape[1])
    model.load_state_dict(item["state"])
    trained_view.clear()
    options = list(metadata["plot_columns"]) + (snp_pc_names if all_snps_label in features else [])
    axis_one.options = options
    axis_one.value = options[0]
    axis_two.options = [("None",None)] + [(name,name) for name in options]
    axis_two.value = options[1] if len(options)>1 else None
    plot_layout(prediction_plot).xaxis.title.text = f"Actual {trait}"
    plot_layout(prediction_plot).yaxis.title.text = f"Predicted {trait}"
    render_training(model,data,features,trait,item["history"],epochs.value,epochs.value,x_tensor,
                    item.get("best_epoch"))
    network_box.children = [make_network_chart(model,x_tensor[torch.from_numpy(masks["Train"]).to(device)],
                                               data[3],features,trait,metadata)]
    outcome = " (stopped early)" if item.get("stopped", False) else ""
    status.value = f"<b>Restored run {item['id']}{outcome}.</b> Validation RMSE: {item['val_rmse']:.2f}."
    if "diagnostic_model_selector" in globals():
        diagnostic_model_selector.value = run_id

run_sort.observe(refresh_first_history, names="value")
run_descending.observe(refresh_first_history, names="value")
run_restore_button.on_click(lambda _: restore_first(run_restore_selector.value))

async def training_yield():
    kernel = get_ipython().kernel if in_colab else None
    parent = kernel.get_parent("shell") if kernel is not None else None
    await asyncio.sleep(0.01)
    if kernel is not None:
        try:
            await kernel.do_one_iteration()
        finally:
            kernel.set_parent(None, parent)

async def train_model(_=None):
    for control in controls:
        control.disabled = True
    stop_button.disabled = False
    status.value = "<b>Training...</b>"
    network_box.children = [widgets.HTML("<p>Training a new model. Its network will appear here shortly.</p>")]
    try:
        await training_yield()
        if training_state["stop_requested"]:
            status.value = "<b>Training stopped before preparation.</b> No run was saved."
            network_box.children = [widgets.HTML("<p>Training stopped before a model was built.</p>")]
            return
        features = list(feature_list.options)
        trait = trait_dropdown.value
        data = make_dataset(features, trait)
        plot_layout(prediction_plot).title.text = "Predicted vs actual yield"
        plot_layout(prediction_plot).xaxis.title.text = f"Actual {trait}"
        plot_layout(prediction_plot).yaxis.title.text = f"Predicted {trait}"
        trained_view.clear()
        metadata = data[5]
        plot_features = list(metadata["plot_columns"])
        if all_snps_label in features:
            plot_features.extend(snp_pc_names)
        axis_one.options = plot_features
        axis_one.value = plot_features[0]
        axis_two.options = [("None", None)] + [(name, name) for name in plot_features]
        axis_two.value = plot_features[1] if len(plot_features) > 1 else None
        x, y, masks, (xm, xs, ym, ys), rows, metadata = data
        x_tensor = torch.from_numpy(x.copy())
        x_tensor.sub_(torch.from_numpy(xm)).div_(torch.from_numpy(xs))
        x_tensor = x_tensor.to(device)
        y_tensor = torch.from_numpy(((y - ym) / ys).reshape(-1, 1)).to(device)
        train_mask = torch.from_numpy(masks["Train"]).to(device)
        val_mask = torch.from_numpy(masks["Validation"]).to(device)
        torch.manual_seed(7)
        model = build_model(neurons.value, layers.value, x.shape[1]).to(device)
        validate_model(model, x.shape[1])
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate.value,
                                     weight_decay=weight_decay.value)
        loss_fn = nn.MSELoss()
        stopper = EarlyStopping(patience=1000)
        history = {"epoch": [], "train": [], "validation": [],
                   "validation_epoch": [], "validation_loss": [],
                   "ranking": {"epoch": [], "Train": [], "Validation": [],
                               "Train trials": [], "Validation trials": []},
                   "ranking_pair": {"epoch": [], "Train genotype": [], "Validation genotype": [],
                                    "Train environment": [], "Validation environment": [],
                                    "Train trials": [], "Validation trials": [],
                                    "Train genotypes": [], "Validation genotypes": []}}
        ranking_rows = {name: [row for row, keep in zip(rows, masks[name]) if keep]
                        for name in ("Train", "Validation")}
        ranking_masks = {"Train": train_mask, "Validation": val_mask}
        total = epochs.value
        progress.max, progress.value = total, 0
        loop = asyncio.get_running_loop()
        next_yield = loop.time()
        for epoch in range(1, total + 1):
            indices = torch.nonzero(train_mask, as_tuple=True)[0]
            indices = indices[torch.randperm(len(indices), device=device)]
            for start in range(0, len(indices), batch_size.value):
                # Yield to the kernel so widget stop messages can be processed.
                if loop.time() >= next_yield:
                    await training_yield()
                    next_yield = loop.time() + 0.05
                if training_state["stop_requested"]:
                    break
                batch = indices[start:start + batch_size.value]
                optimizer.zero_grad()
                loss = loss_fn(model(x_tensor[batch]), y_tensor[batch])
                if not torch.isfinite(loss):
                    raise ValueError("Training produced a non-finite loss. Try a smaller learning rate.")
                loss.backward()
                optimizer.step()
            with torch.no_grad():
                val_loss = float(loss_fn(model(x_tensor[val_mask]), y_tensor[val_mask]))
            early_stopped = stopper.update(model, val_loss, epoch)
            history["validation_epoch"].append(epoch)
            history["validation_loss"].append(val_loss)
            history["ranking"]["epoch"].append(epoch)
            history["ranking_pair"]["epoch"].append(epoch)
            for name in ("Train", "Validation"):
                with torch.no_grad():
                    estimate = model(x_tensor[ranking_masks[name]]).cpu().numpy().ravel() * ys + ym
                scores = grouped_ranking_metrics(ranking_rows[name], y[masks[name]], estimate)
                history["ranking"][name].append(scores["genotype spearman"])
                history["ranking"][name + " trials"].append(scores["trials"])
                history["ranking_pair"][name + " genotype"].append(scores["genotype pearson"])
                history["ranking_pair"][name + " environment"].append(scores["environment pearson"])
                history["ranking_pair"][name + " trials"].append(scores["trials"])
                history["ranking_pair"][name + " genotypes"].append(scores["genotypes"])
            if epoch == 1 or epoch % 10 == 0 or epoch == total or training_state["stop_requested"] or early_stopped:
                with torch.no_grad():
                    train_loss = float(loss_fn(model(x_tensor[train_mask]), y_tensor[train_mask]))
                if not np.isfinite(train_loss):
                    raise ValueError("Training produced a non-finite loss. Try a smaller learning rate.")
                history["epoch"].append(epoch)
                history["train"].append(train_loss)
                history["validation"].append(val_loss)
                render_training(model, data, features, trait, history, epoch, total, x_tensor,
                                stopper.best_epoch)
                progress.value = epoch
            if training_state["stop_requested"] or early_stopped:
                break
        stopper.restore(model)
        render_training(model, data, features, trait, history, epoch, epoch, x_tensor,
                        stopper.best_epoch)
        network_box.children = [make_network_chart(model, x_tensor[train_mask], data[3], features, trait, metadata)]
        val_rmse = float(np.sqrt(stopper.best_loss) * ys)
        metrics = run_metrics(model, data, trait, x_tensor)
        first_history.append({"id":len(first_history)+1, "features":features, "trait":trait,
            "settings":(neurons.value,layers.value,epochs.value,batch_size.value,learning_rate.value,weight_decay.value),
            "val_rmse":val_rmse, "metrics":metrics, "history":history, "stopped":training_state["stop_requested"],
            "best_epoch":stopper.best_epoch, "best_validation_mse":stopper.best_loss,
            "state":{key:value.detach().cpu().clone() for key,value in model.state_dict().items()}})
        refresh_first_history(select_latest=True)
        details = ""
        snp_inputs = [name for name in features if name in snp_features or name == all_snps_label]
        if snp_inputs:
            missing = np.asarray([any(
                not np.isfinite(multi_cache["genetic"][row["genotype"]]).all()
                if name == all_snps_label else
                not np.isfinite(multi_cache["genetic"][row["genotype"]][snp_features[name]])
                for name in snp_inputs)
                                  for row in rows])
            counts = ", ".join(f"{name}: {int(mask.sum())} ({int((mask & missing).sum())} imputed)"
                               for name, mask in masks.items())
            excluded = sum(trait in row["traits"] and row["genotype"] not in multi_cache["genetic"] and split_name(row) != "Not used"
                           for row in observations)
            details = f"<br>Matched records — {counts}; {excluded} split records lack SNP data and were excluded."
        outcome = "Training stopped; partial run saved" if training_state["stop_requested"] else "Training complete"
        if early_stopped:
            stopping_note = "Early-stopping patience reached."
        elif training_state["stop_requested"]:
            stopping_note = "Stopped manually before the early-stopping patience was reached."
        else:
            stopping_note = f"Reached the {epoch}-epoch budget; patience (1,000 epochs) was not reached."
        status.value = (f"<b>{outcome}.</b> Validation RMSE: {val_rmse:.2f} "
                        f"({html.escape(trait)}). {stopping_note} Restored best epoch "
                        f"{stopper.best_epoch}/{epoch}.{details}")
        return model, data, history
    except Exception as exc:
        status.value = f"<b>Training failed:</b> {html.escape(str(exc))}"
        raise
    finally:
        stop_button.disabled = True
        training_state["mode"] = None
        for control in controls:
            control.disabled = False
        run_restore_button.disabled = not bool(first_history)


def training_finished(task):
    if task.cancelled():
        status.value = "<b>Training cancelled.</b> Rerun the playground cell before training again."
    else:
        exc = task.exception()
        if exc is not None:
            status.value = f"<b>Training failed:</b> {html.escape(str(exc))}"

def start_training(_=None):
    task = training_state["task"]
    if task is not None and not task.done():
        return
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError as exc:
        status.value = ("<b>Training could not start:</b> the notebook has no active asyncio event loop. "
                        "Rerun this cell, then try again.")
        raise RuntimeError("The training button requires the active notebook event loop.") from exc
    training_state["stop_requested"] = False
    training_state["mode"] = "training"
    train_button.disabled = True
    status.value = "<b>Starting training...</b>"
    task = loop.create_task(train_model())
    training_state["task"] = task
    task.add_done_callback(training_finished)
    if in_colab:
        import nest_asyncio
        nest_asyncio.apply(loop)
        loop.run_until_complete(task)

def stop_training(_=None):
    task = training_state["task"]
    if task is not None and not task.done():
        training_state["stop_requested"] = True
        stop_button.disabled = True
        status.value = "<b>Stopping training...</b>"

def trait_changed(change):
    if trained_view:
        status.value = ("<b>Trait changed.</b> Retrain to update the plots; "
                        "the plots still show the previous trait.")

trait_dropdown.observe(trait_changed, names="value")
train_button.on_click(start_training)
stop_button.on_click(stop_training)
debug_notice = widgets.HTML(
    "<b>Debug mode: synthetic SNPs; results are for debugging only.</b>"
    if debug_dummy_snps else "")
control_heading = widgets.HTML(
    '<div style="padding:12px 0 8px;color:#17334b;font-weight:700;font-size:17px">'
    'Your experiment</div>')
selectors = widgets.VBox([
    widgets.HBox([ec_dropdown, add_feature], layout=widgets.Layout(flex_flow="row wrap")),
    widgets.HBox([feature_list, remove_feature], layout=widgets.Layout(flex_flow="row wrap")),
    trait_dropdown])
tuning = widgets.HBox([widgets.VBox([learning_rate, weight_decay, batch_size]),
                       widgets.VBox([neurons, layers, epochs])],
    layout=widgets.Layout(flex_flow="row wrap", width="100%"))
action = widgets.HBox([train_button, stop_button, progress],
    layout=widgets.Layout(flex_flow="row wrap", align_items="center", grid_gap="12px"))
display_panel(widgets.VBox([debug_notice, control_heading, selectors, tuning, action, status,
                          plot,
                          widgets.HBox([axis_one, axis_two], layout=widgets.Layout(flex_flow="row wrap")),
                          axis_status, feature_box, prediction_plot,
                          widgets.HTML("<b>Model records</b>"),
                          widgets.HBox([run_sort, run_descending]),
                          widgets.HBox([run_restore_selector, run_restore_button],
                                       layout=widgets.Layout(flex_flow="row wrap")),
                          widgets.Box([run_table], layout=widgets.Layout(overflow="auto", width="100%"))],
    layout=widgets.Layout(width="100%", max_width="1200px", padding="8px 16px 16px",
                          border="1px solid #e1ecee")))

#@title Explore the trained network
display_panel(network_box)
'''

diagnostic_playground_source = r'''
#@title Select a trained model and inspect its diagnostics
import html

diagnostic_model_selector = widgets.Dropdown(
    options=[], description="Trained model",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="680px", max_width="100%"))
diagnostic_status = widgets.HTML("Train a model in the corresponding playground to see its diagnostics.")

def diagnostic_figure(title, y_title, traces):
    figure = figure_widget()
    for trace in traces:
        figure.add_scatter(**trace)
    chart_layout(figure, 390)
    figure.update_layout(title=title)
    figure.update_xaxes(title_text="Epoch")
    figure.update_yaxes(title_text=y_title, range=[-1.05, 1.05])
    return figure

within_trial_diagnostic = diagnostic_figure(
    "Within-trial correlation",
    "Mean within-trial correlation (higher is better)",
    [dict(x=[], y=[], mode="lines", name=f"{split} | Pearson",
          line=dict(color=color, width=3), customdata=[],
          hovertemplate="Epoch %{x}<br>Within-trial correlation (Pearson r): %{y:.3f}<br>Defined trials: %{customdata}<extra>" + split + "</extra>")
     for split, color in (("Train", split_colors["Train"]),
                          ("Validation", split_colors["Validation"]))]
    + [dict(x=[], y=[], mode="lines", name=f"{split} | Spearman",
            line=dict(color=color, width=2, dash="dash"), customdata=[],
            hovertemplate="Epoch %{x}<br>Within-trial correlation (Spearman rho): %{y:.3f}<br>Defined trials: %{customdata}<extra>" + split + "</extra>")
       for split, color in (("Train", split_colors["Train"]),
                            ("Validation", split_colors["Validation"]))])
within_environment_diagnostic = diagnostic_figure(
    "Within-genotype correlation",
    "Mean within-genotype correlation (Pearson r; higher is better)",
    [dict(x=[], y=[], mode="lines+markers", name=split,
          line=dict(color=split_colors[split], width=3), customdata=[],
          hovertemplate="Epoch %{x}<br>Within-genotype correlation (Pearson r): %{y:.3f}<br>Defined genotypes: %{customdata}<extra>" + split + "</extra>")
     for split in ("Train", "Validation")])
accuracy_tradeoff = figure_widget()
for split in ("Train", "Validation"):
    accuracy_tradeoff.add_scatter(
        x=[], y=[], mode="lines+markers", name=split,
        connectgaps=False, line=dict(color=split_colors[split], width=2),
        marker=dict(size=7), customdata=[],
        hovertemplate="Epoch %{customdata[0]}<br>Within-genotype correlation (Pearson r): %{x:.3f}"
            "<br>Within-trial correlation (Pearson r): %{y:.3f}<br>Defined trials: %{customdata[1]}"
            "<br>Defined genotypes: %{customdata[2]}<extra>" + split + "</extra>")
chart_layout(accuracy_tradeoff, 390)
accuracy_tradeoff.add_scatter(x=[], y=[], mode="lines+markers", name="Validation Pareto front",
    line=dict(color="#293e73", width=3, dash="dash"), marker=dict(size=10, symbol="diamond"),
    customdata=[], hovertemplate="Pareto-optimal validation epoch %{customdata}"
        "<br>Within-genotype correlation (Pearson r): %{x:.3f}"
        "<br>Within-trial correlation (Pearson r): %{y:.3f}<extra>Validation Pareto front</extra>")
accuracy_tradeoff.update_layout(title="Within-trial vs within-genotype correlation · validation Pareto front")
accuracy_tradeoff.update_xaxes(title_text="Within-genotype correlation (Pearson r)", range=[-1.05, 1.05])
accuracy_tradeoff.update_yaxes(title_text="Within-trial correlation (Pearson r)", range=[-1.05, 1.05])

def clear_posthoc_diagnostics():
    for figure in (within_trial_diagnostic, within_environment_diagnostic, accuracy_tradeoff):
        with figure.batch_update():
            figure.update_layout(shapes=[], overwrite=True)
            for trace in figure.data:
                trace.x, trace.y, trace.customdata = [], [], []

def validation_correlation_peak(epochs, values):
    candidates = [(epoch, value) for epoch, value in zip(epochs, values)
                  if value is not None and np.isfinite(value)]
    return max(candidates, key=lambda point: point[1], default=None)

def validation_pareto_front(epochs, within_genotype, within_trial):
    candidates = [(epoch, x, y) for epoch, x, y in zip(epochs, within_genotype, within_trial)
                  if x is not None and y is not None and np.isfinite(x) and np.isfinite(y)]
    candidates.sort(key=lambda point: (-point[1], -point[2], point[0]))
    frontier = []
    best_y = -np.inf
    for point in candidates:
        if point[2] > best_y:
            frontier.append(point)
            best_y = point[2]
    return frontier[::-1]

def update_posthoc_diagnostics(change=None):
    run_id = diagnostic_model_selector.value
    item = next((entry for entry in first_history if entry["id"] == run_id), None)
    if item is None:
        clear_posthoc_diagnostics()
        diagnostic_status.value = "Train a model in the corresponding playground to see its diagnostics."
        return
    history = item["history"]
    ranking = history.get("ranking_pair", {})
    epochs = ranking.get("epoch", [])
    for index, split in enumerate(("Train", "Validation")):
        counts = ranking.get(split + " trials", [])
        within_trial_diagnostic.data[index].x = epochs
        within_trial_diagnostic.data[index].y = ranking.get(split + " genotype", [])
        within_trial_diagnostic.data[index].customdata = counts
        within_trial_diagnostic.data[index + 2].x = epochs
        within_trial_diagnostic.data[index + 2].y = history.get("ranking", {}).get(split, [])
        within_trial_diagnostic.data[index + 2].customdata = history.get("ranking", {}).get(
            split + " trials", [])
        within_environment_diagnostic.data[index].x = epochs
        within_environment_diagnostic.data[index].y = ranking.get(split + " environment", [])
        within_environment_diagnostic.data[index].customdata = ranking.get(split + " genotypes", [])
        accuracy_tradeoff.data[index].x = ranking.get(split + " environment", [])
        accuracy_tradeoff.data[index].y = ranking.get(split + " genotype", [])
        accuracy_tradeoff.data[index].customdata = [
            [epoch, trial_count, genotype_count]
            for epoch, trial_count, genotype_count in zip(
                epochs, ranking.get(split + " trials", []), ranking.get(split + " genotypes", []))]
    for figure in (within_trial_diagnostic, within_environment_diagnostic, accuracy_tradeoff):
        figure.update_layout(shapes=[], overwrite=True)
    frontier = validation_pareto_front(epochs, ranking.get("Validation environment", []),
                                       ranking.get("Validation genotype", []))
    accuracy_tradeoff.data[2].x = [point[1] for point in frontier]
    accuracy_tradeoff.data[2].y = [point[2] for point in frontier]
    accuracy_tradeoff.data[2].customdata = [point[0] for point in frontier]
    for key, label, figure, color in (
        ("genotype", "within-trial", within_trial_diagnostic, "#407fa5"),
        ("environment", "within-genotype", within_environment_diagnostic, "#7864a8"),
    ):
        peak = validation_correlation_peak(epochs, ranking.get("Validation " + key, []))
        if peak is not None:
            epoch, _ = peak
            name = f"Max validation {label} Pearson r (epoch {epoch})"
            line = dict(color=color, width=2, dash="dot")
            figure.add_vline(x=epoch, line=line, name=name, showlegend=True)
    stopped = " (stopped early)" if item["stopped"] else ""
    for figure in (within_trial_diagnostic, within_environment_diagnostic, accuracy_tradeoff):
        refresh_colab_plots(figure)
    diagnostic_status.value = (
        f"<b>Run {item['id']}{stopped}</b> | {html.escape(item['trait'])} | "
        f"{len(epochs)} epochs. Training and validation records only; test labels are not shown.")

def refresh_posthoc_models(change=None, select_latest=False):
    if not first_history:
        diagnostic_model_selector.options = []
        clear_posthoc_diagnostics()
        diagnostic_status.value = "Train a model in the corresponding playground to see its diagnostics."
        return
    previous = diagnostic_model_selector.value
    options = [(f"Run {item['id']} | {item['trait']} | {', '.join(item['features'])}", item["id"])
               for item in first_history]
    diagnostic_model_selector.options = options
    run_ids = {item["id"] for item in first_history}
    diagnostic_model_selector.value = (first_history[-1]["id"]
        if select_latest or previous not in run_ids else previous)
    update_posthoc_diagnostics()

diagnostic_model_selector.observe(update_posthoc_diagnostics, names="value")
refresh_posthoc_models()
display_panel(widgets.VBox([
    diagnostic_model_selector, diagnostic_status,
    within_trial_diagnostic, within_environment_diagnostic, accuracy_tradeoff,
], layout=widgets.Layout(width="100%", max_width="1200px", padding="8px 16px 16px",
                         border="1px solid #e1ecee")))
'''

test_playground_source = r'''
#@title Reveal the test scores of your saved models
test_reveal_button = widgets.Button(description="Reveal test scores", icon="table",
    layout=widgets.Layout(width="200px"))
test_comparison_status = widgets.HTML()
test_comparison_table = widgets.HTML()

def refresh_test_comparison():
    test_reveal_button.disabled = not bool(first_history)
    test_comparison_table.value = ""
    test_comparison_status.value = (
        "<p>Test scores are hidden. Finish model selection, then reveal all saved runs.</p>"
        if first_history else "<p>Train a model in the corresponding playground before revealing test scores.</p>")

def saved_test_metrics(item):
    data = make_dataset(item["features"], item["trait"])
    x, y, masks, (xm, xs, ym, ys), rows, metadata = data
    x_tensor = torch.from_numpy(x.copy())
    x_tensor.sub_(torch.from_numpy(xm)).div_(torch.from_numpy(xs))
    x_tensor = x_tensor.to(device)
    model = build_model(item["settings"][0], item["settings"][1], x.shape[1]).to(device)
    model.load_state_dict(item["state"])
    model.eval()
    scores = run_metrics(model, data, item["trait"], x_tensor, splits=("Test",))
    scores["Test records"] = int(masks["Test"].sum())
    return scores

def reveal_test_scores(_=None):
    test_comparison_table.value = ""
    if not first_history:
        refresh_test_comparison()
        return
    test_reveal_button.disabled = True
    test_comparison_status.value = "<p>Evaluating saved checkpoints on the test set...</p>"
    try:
        table_rows = []
        for item in first_history:
            scores = saved_test_metrics(item)
            inputs = "<br>".join(html.escape(name) for name in item["features"])
            partial = "<br>Stopped manually" if item["stopped"] else ""
            cells = [f"<b>Run {item['id']}</b>{partial}",
                     f"<b>{html.escape(item['trait'])}</b><br>{inputs}",
                     str(scores["Test records"]), f"{scores['Test rmse']:.3f}"]
            for metric, count in (("pearson", "trials"), ("environment pearson", "genotypes")):
                value = scores["Test " + metric]
                text = "undefined" if value is None else f"{value:.3f}"
                cells.append(f"{text}<br>{scores['Test ' + count]} groups")
            table_rows.append("<tr>" + "".join(
                f"<td style='padding:9px 10px;border-bottom:1px solid #e6edef;vertical-align:top'>{cell}</td>"
                for cell in cells) + "</tr>")
        headings = ("Run", "Trait / model inputs", "Test records", "Test plot-level RMSE",
                    "Test within-trial correlation (Pearson r)", "Test within-genotype correlation (Pearson r)")
        header = "<tr>" + "".join(
            f"<th style='padding:9px 10px;background:#e8f2f1;text-align:left'>{label}</th>"
            for label in headings) + "</tr>"
        test_comparison_table.value = (
            "<div style='width:100%;overflow-x:auto'><table style='width:100%;min-width:900px;"
            "border-collapse:collapse;font:13px Arial,sans-serif;color:#17334b'>"
            "<thead>" + header + "</thead><tbody>" + "".join(table_rows) + "</tbody></table></div>")
        test_comparison_status.value = (
            f"<p>Test scores revealed for {len(first_history)} saved runs using their best-validation checkpoints.</p>")
    except Exception as exc:
        test_comparison_status.value = f"<b>Test evaluation failed:</b> {html.escape(str(exc))}"
        raise
    finally:
        test_reveal_button.disabled = not bool(first_history)

test_reveal_button.on_click(reveal_test_scores)
refresh_test_comparison()
display_panel(widgets.VBox([test_reveal_button, test_comparison_status, test_comparison_table],
    layout=widgets.Layout(width="100%", max_width="1200px", padding="8px 16px 16px",
                          border="1px solid #e1ecee")))
'''

def playground_namespace(splitter, label):
    namespace = globals().copy()
    namespace["split_name"] = splitter
    namespace["dataset_label"] = label
    return namespace


## 01 - Complete dataset: model-training playground

This section uses the **complete dataset** split described in Section 00. Controls and saved runs belong only to this dataset.

Choose a trait, inputs, and settings, then select **Train / retrain**. The curve marks the lowest-validation-MSE checkpoint, which is restored after training. Early-stopping patience is 1,000 epochs, above the slider maximum of 600; runs normally end at the selected epoch budget unless you stop them manually. Test results are reserved for a final evaluation.


In [ ]:
#@title Complete dataset: train your models
complete_playground = playground_namespace(complete_split_name, "Complete dataset")
exec(compile(training_playground_source, "complete-training", "exec"), complete_playground)


## 02 - Complete dataset: diagnosing issues

Select a model trained in Section 01. The plots show training and validation within-trial correlation (genotype differences within each trial) and within-genotype correlation (environment differences for each genotype) over epochs. No model is trained here; test scores remain hidden until Section 03. Ideally, these would converge at similar speeds for balanced multimodal training.

Dotted vertical lines mark the first epoch with the highest defined validation Pearson correlation for each measure. The joint plot highlights the **validation Pareto front**: epochs for which no other epoch has both correlations at least as high and one strictly higher. Both correlations are maximized; hovering shows the epoch. Duplicate correlation pairs show their earliest epoch, and pairs with either correlation undefined are omitted. The connecting line is a visual guide, not an evaluated mixture of models. Neither the front nor the maximum markers change the restored best-validation-MSE checkpoint.


In [ ]:
#@title Complete dataset: inspect model diagnostics
exec(compile(diagnostic_playground_source, "complete-diagnostics", "exec"), complete_playground)


## 03 - Complete dataset: compare models on the test set

Only models trained in Section 01 are evaluated here, on this dataset's held-out genotypes in 2019.

Finish choosing your models using training and validation results, then click **Reveal test scores** to compare all saved runs on the held-out 2019 test set. Each model uses its restored best-validation checkpoint; no retraining is performed. If you train another model, click the button again to update the comparison.

**Lower plot-level RMSE is better; higher Pearson correlations are better.** Within-trial correlation measures how well a model predicts genotype differences within each trial; within-genotype correlation measures environment differences for each genotype. A trial is a year × location × management combination. Correlations are averaged over groups with at least two records and non-constant observations and predictions; otherwise they are shown as undefined. Record and valid-group counts are included because models using SNPs may exclude cultivars without genetic data. Compare scores only for the same trait and consider differences in coverage. Keep test results for this final assessment, not further model tuning.


In [ ]:
#@title Complete dataset: reveal test scores
exec(compile(test_playground_source, "complete-test", "exec"), complete_playground)


## 04 - Sparse cohort dataset: model-training playground

This section uses the **sparse cohort dataset** split described in Section 00. Controls and saved runs belong only to this dataset.

Choose a trait, inputs, and settings, then select **Train / retrain**. The curve marks the lowest-validation-MSE checkpoint, which is restored after training. Early-stopping patience is 1,000 epochs, above the slider maximum of 600; runs normally end at the selected epoch budget unless you stop them manually. Test results are reserved for a final evaluation.


In [ ]:
#@title Sparse cohort dataset: train your models
sparse_playground = playground_namespace(split_name, "Sparse cohort dataset")
exec(compile(training_playground_source, "sparse-training", "exec"), sparse_playground)


## 05 - Sparse cohort dataset: diagnosing issues

Select a model trained in Section 04. The plots show training and validation within-trial correlation (genotype differences within each trial) and within-genotype correlation (environment differences for each genotype) over epochs. No model is trained here; test scores remain hidden until Section 06. Ideally, these would converge at similar speeds for balanced multimodal training.

Dotted vertical lines mark the first epoch with the highest defined validation Pearson correlation for each measure. The joint plot highlights the **validation Pareto front**: epochs for which no other epoch has both correlations at least as high and one strictly higher. Both correlations are maximized; hovering shows the epoch. Duplicate correlation pairs show their earliest epoch, and pairs with either correlation undefined are omitted. The connecting line is a visual guide, not an evaluated mixture of models. Neither the front nor the maximum markers change the restored best-validation-MSE checkpoint.


In [ ]:
#@title Sparse cohort dataset: inspect model diagnostics
exec(compile(diagnostic_playground_source, "sparse-diagnostics", "exec"), sparse_playground)


## 06 - Sparse cohort dataset: compare models on the test set

Only models trained in Section 04 are evaluated here, on this dataset's held-out genotypes in 2019.

Finish choosing your models using training and validation results, then click **Reveal test scores** to compare all saved runs on the held-out 2019 test set. Each model uses its restored best-validation checkpoint; no retraining is performed. If you train another model, click the button again to update the comparison.

**Lower plot-level RMSE is better; higher Pearson correlations are better.** Within-trial correlation measures how well a model predicts genotype differences within each trial; within-genotype correlation measures environment differences for each genotype. A trial is a year × location × management combination. Correlations are averaged over groups with at least two records and non-constant observations and predictions; otherwise they are shown as undefined. Record and valid-group counts are included because models using SNPs may exclude cultivars without genetic data. Compare scores only for the same trait and consider differences in coverage. Keep test results for this final assessment, not further model tuning.


In [ ]:
#@title Sparse cohort dataset: reveal test scores
exec(compile(test_playground_source, "sparse-test", "exec"), sparse_playground)


## 07 - Credits and references

- **BRIWECS field, soil, coordinates, and weather data:** [dataset](https://doi.org/10.6084/m9.figshare.27910269.v1) and [paper](https://doi.org/10.1038/s41597-024-04332-7). The dataset is [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/); credit the authors and note changes.
- **SNPs:** [Lichthardt et al. (2020)](https://doi.org/10.3389/fpls.2019.01771), Supplementary Data Sheet 2. Check its notices before redistributing the workbook.
- **Weather:** German Weather Service [DWD terms](https://opendata.dwd.de/climate_environment/CDC/Terms_of_use.pdf) (CC BY 4.0).